In [1]:
#!/usr/bin/env python
# coding: utf-8
# ★ PATCH v3 — 산업별현황 + 404억제 (2026-05-02)

# In[ ]:


import yfinance as yf
import ta
import numpy as np
import pandas as pd
import platform
import smtplib
import os
import requests
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
from apscheduler.schedulers.blocking import BlockingScheduler
from IPython.display import display, HTML
import logging, warnings, contextlib
# ── yfinance HTTP 오류 전면 억제 ──────────────────────────
# yfinance 포함 모든 외부 라이브러리 오류 메시지 전면 억제
for _lg in ['yfinance', 'yfinance.base', 'yfinance.utils',
            'yfinance.ticker', 'yfinance.scrapers', 'yfinance.data',
            'peewee', 'urllib3', 'requests', 'root']:
    logging.getLogger(_lg).setLevel(logging.CRITICAL)
logging.disable(logging.ERROR)            # ERROR 이하 전역 비활성 (404 포함)
warnings.filterwarnings('ignore')         # 모든 경고 억제
# ────────────────────────────────────────────────────────

# ════════════════════════════════════════════
#  ★ 사용자 설정 — 여기만 수정하세요
# ════════════════════════════════════════════
import os
try:
    import streamlit as st
    _HAS_ST = True
except ImportError:
    _HAS_ST = False
EMAIL_FROM     = "dltkdals15319@gmail.com"
EMAIL_PASSWORD = "kiobsnqaugxjaucc"
EMAIL_TO       = "dltkdals15319@gmail.com"
GSHEET_KEY_FILE = os.environ.get("GSHEET_KEY_FILE", "stately-transit-494900-i8-524c44e2e62d.json")
GSHEET_ID            = os.environ.get("GSHEET_ID", "1Yho-uNOCeMzeLZEjOudoJVMzlwBDLhTlmilE3W7HSfU")
GSHEET_SHEET         = os.environ.get("GSHEET_SHEET", "📈주식현황상세")  # 주식현황
GSHEET_SHEET_DEPOSIT = os.environ.get("GSHEET_SHEET_DEPOSIT", "🌱예적금내역")  # 예적금
GSHEET_SHEET_CASH    = os.environ.get("GSHEET_SHEET_CASH",    "💰입출금통장")  # 입출금

PORTFOLIO_FILE  = "portfolio.csv"
SCHEDULE_HOUR   = 0
SCHEDULE_MINUTE = 34

SCAN_BUY_THRESHOLD = 3  # 기본 신호 최소 개수
SCAN_WEEKLY_MIN    = 2
PORT_BUY_THRESHOLD = 3

# ════════════════════════════════════════════
#  신규 — yfinance 자동 티커/섹터 수집
# ════════════════════════════════════════════

def get_sp500_tickers_auto():
    """
    S&P500 구성종목 자동 수집
    방법 1: yfinance SPY funds_data (최신 yfinance 0.2.x+)
    방법 2: yfinance IVV/VOO funds_data
    방법 3: Wikipedia requests 파싱
    방법 4: 하드코딩 폴백
    """
    # 방법 1: SPY ETF 구성종목
    for etf in ["SPY", "IVV", "VOO"]:
        try:
            ticker = yf.Ticker(etf)
            # yfinance 0.2.x funds_data 방식
            fd = ticker.funds_data
            if hasattr(fd, 'top_holdings') and fd.top_holdings is not None:
                tickers = fd.top_holdings.index.tolist()
                if len(tickers) > 50:
                    print(f"  S&P500 로드 완료 ({etf} funds_data): {len(tickers)}종목")
                    return tickers
        except Exception:
            pass

    # 방법 2: yfinance info → holdings (구버전 호환)
    try:
        spy = yf.Ticker("SPY")
        info = spy.info
        if "holdings" in info and len(info["holdings"]) > 50:
            tickers = [h["symbol"] for h in info["holdings"]]
            print(f"  S&P500 로드 완료 (SPY info): {len(tickers)}종목")
            return tickers
    except Exception:
        pass

    # 방법 3: Wikipedia requests 파싱
    try:
        url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                  "AppleWebKit/537.36"}
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        tbl = pd.read_html(resp.text, header=0)[0]
        tickers = tbl["Symbol"].str.replace(".", "-", regex=False).tolist()
        print(f"  S&P500 로드 완료 (Wikipedia): {len(tickers)}종목")
        return tickers
    except Exception as e:
        print(f"  ⚠ S&P500 Wikipedia 파싱 실패: {e}")

    # 방법 4: 폴백 하드코딩
    print(f"  ⚠ S&P500 자동 로드 전체 실패 — 하드코딩 사용 ({len(SP500_FALLBACK)}종목)")
    return SP500_FALLBACK


def get_nasdaq100_tickers_auto():
    """
    나스닥100 구성종목 자동 수집
    방법 1: yfinance QQQ funds_data
    방법 2: Wikipedia requests 파싱
    방법 3: 하드코딩 폴백
    """
    # 방법 1: QQQ ETF 구성종목
    for etf in ["QQQ", "QQQM"]:
        try:
            ticker = yf.Ticker(etf)
            fd = ticker.funds_data
            if hasattr(fd, 'top_holdings') and fd.top_holdings is not None:
                tickers = fd.top_holdings.index.tolist()
                if len(tickers) > 50:
                    print(f"  나스닥100 로드 완료 ({etf} funds_data): {len(tickers)}종목")
                    return tickers
        except Exception:
            pass

    # 방법 2: Wikipedia 파싱
    try:
        url = "https://en.wikipedia.org/wiki/Nasdaq-100"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                  "AppleWebKit/537.36"}
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        tables = pd.read_html(resp.text, header=0)
        for tbl in tables:
            cols = [c.lower() for c in tbl.columns]
            if "ticker" in cols or "symbol" in cols:
                col = "Ticker" if "Ticker" in tbl.columns else "Symbol"
                tickers = tbl[col].str.replace(".", "-", regex=False).tolist()
                print(f"  나스닥100 로드 완료 (Wikipedia): {len(tickers)}종목")
                return tickers
    except Exception as e:
        print(f"  ⚠ 나스닥100 Wikipedia 파싱 실패: {e}")

    # 방법 3: 폴백 하드코딩
    print(f"  ⚠ 나스닥100 자동 로드 전체 실패 — 하드코딩 사용 ({len(NASDAQ100_FALLBACK)}종목)")
    return NASDAQ100_FALLBACK


def get_ticker_sector_map_auto(tickers):
    """
    yfinance info에서 종목별 섹터 자동 수집
    sector 필드 → SPDR ETF 코드로 매핑
    """
    SECTOR_TO_ETF = {
        "Technology":             "XLK",
        "Financial Services":     "XLF",
        "Healthcare":             "XLV",
        "Health Care":            "XLV",
        "Energy":                 "XLE",
        "Industrials":            "XLI",
        "Communication Services": "XLC",
        "Consumer Cyclical":      "XLY",
        "Consumer Defensive":     "XLP",
        "Basic Materials":        "XLB",
        "Utilities":              "XLU",
        "Real Estate":            "XLRE",
    }

    print(f"\n  [섹터 맵 자동 구축] {len(tickers)}종목 섹터 정보 수집 중...")
    sector_map = {}
    batch_size = 50   # yfinance 배치 다운로드

    # yfinance download로 배치 처리 (빠름)
    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        for t in batch:
            try:
                info = yf.Ticker(t).fast_info
                # fast_info는 sector 없음 → 개별 info 호출
                sector_str = yf.Ticker(t).info.get("sector", "")
                etf = SECTOR_TO_ETF.get(sector_str, "")
                if etf:
                    sector_map[t] = etf
            except Exception:
                pass
        if (i + batch_size) % 100 == 0:
            print(f"    섹터 맵 진행: {min(i+batch_size, len(tickers))}/{len(tickers)}")

    print(f"  섹터 맵 완료: {len(sector_map)}종목 매핑됨")
    return sector_map


# ── 폴백 하드코딩 (자동 로드 실패 시) ────────
SP500_FALLBACK = [
    "AAPL","MSFT","NVDA","AVGO","AMD","QCOM","INTC","MU","AMAT","LRCX",
    "KLAC","TXN","ADI","MRVL","NXPI","MPWR","FTNT","CDNS","SNPS","CRM",
    "ORCL","NOW","INTU","ADBE","PANW","CRWD","ZS","DDOG","SNOW","WDAY",
    "META","GOOGL","GOOG","NFLX","DIS","CMCSA","T","VZ","TMUS","PARA",
    "AMZN","TSLA","HD","MCD","NKE","SBUX","LOW","TJX","BKNG","MAR",
    "PG","KO","PEP","WMT","COST","PM","MO","GIS","CLX","CL",
    "JPM","BAC","WFC","GS","MS","C","BLK","SCHW","AXP","COF",
    "JNJ","UNH","PFE","ABBV","MRK","LLY","BMY","AMGN","GILD","REGN",
    "GE","CAT","BA","HON","RTX","LMT","NOC","UPS","FDX","CSX",
    "XOM","CVX","COP","SLB","EOG","DVN","HAL","VLO","MPC","OXY",
    "LIN","APD","SHW","PPG","NEM","FCX","NUE","ALB","MOS","CF",
    "NEE","DUK","SO","D","AEP","EXC","XEL","ED","PEG","AWK",
    "AMT","PLD","CCI","EQIX","PSA","O","SPG","WELL","EQR","AVB",
]

NASDAQ100_FALLBACK = [
    "AAPL","MSFT","NVDA","AMZN","META","GOOGL","GOOG","TSLA","AVGO","COST",
    "NFLX","AMD","PEP","ADBE","CSCO","INTC","QCOM","TXN","AMGN","HON",
    "INTU","AMAT","SBUX","GILD","MDLZ","REGN","LRCX","VRTX","KLAC","SNPS",
    "CDNS","ORLY","MELI","PANW","MRVL","ADP","FTNT","CRWD","DXCM","ABNB",
    "ZS","TEAM","WDAY","ODFL","PCAR","BIIB","FAST","DDOG","ON","VRSK",
]

# ════════════════════════════════════════════
#  SPDR 섹터 ETF 정의 (분석용)
# ════════════════════════════════════════════
SECTOR_ETFS = {
    "XLK":  "기술 (Technology)",
    "XLF":  "금융 (Financials)",
    "XLV":  "헬스케어 (Health Care)",
    "XLE":  "에너지 (Energy)",
    "XLI":  "산업재 (Industrials)",
    "XLC":  "커뮤니케이션 (Communication)",
    "XLY":  "임의소비재 (Consumer Discretionary)",
    "XLP":  "필수소비재 (Consumer Staples)",
    "XLB":  "소재 (Materials)",
    "XLU":  "유틸리티 (Utilities)",
    "XLRE": "부동산 (Real Estate)",
}

# ════════════════════════════════════════════════════════
#  전략 파라미터 / 섹터별 전략 배정
# ════════════════════════════════════════════════════════
STRATEGY_PARAMS = {
    "A": {"weekly_min": 0, "need_cloud": False},
    "C": {"weekly_min": 2, "need_cloud": False},
    "D": {"weekly_min": 2, "need_cloud": True },
}

SECTOR_STRATEGY = {
    "XLK": "C", "XLC": "C", "XLY": "C", "XLV": "C", "XLP": "C",
    "XLF": "D",
}
DEFAULT_STRATEGY = "A"

# TICKER_SECTOR: 종목 → 섹터ETF 매핑 (스캔 시 sector_map 으로 동적 생성)
TICKER_SECTOR = {}   # run_market_scan() 호출 전 sector_map 기반으로 채워짐

# ════════════════════════════════════════════════════════
#  벨류체인 투자 분석 DB (NVIDIA / Google / SpaceX / Tesla)
# ════════════════════════════════════════════════════════
VALUE_CHAIN_DB = {
    "NVIDIA": {
        "display": "NVIDIA Corporation",
        "ticker": "NVDA",
        "accent": "#76b900",
        "entries": [
            # (분야, 기업명, 티커, 투자금액USD, 관계유형, 설명, 투자시점, 출처)
            # ── 전력/반도체 ──
            ("전력반도체 / 800V Sidecar",   "Navitas Semiconductor",     "NVTS",  20_000_000,  "전략투자",    "GaN 전력 IC — H100 Sidecar 모듈",             "2022.06", "SEC Form 4 / Bloomberg"),
            ("전력관리 / 800V VRM",          "Monolithic Power Systems",  "MPWR",  None,        "핵심공급사",  "H100/H200/GB200 VRM 독점 공급",               "2020~",   "MPWR IR / Anandtech 분석"),
            ("칩 아키텍처 (ARM)",            "Arm Holdings",              "ARM",   None,        "지분보유",    "GPU 아키텍처 기반 (~1% 지분, IPO 참여)",       "2023.09", "ARM IPO 투자설명서 / Bloomberg"),
            # ── 광통신 / 인터커넥트 ──
            ("광통신 부품 (EML 레이저)",     "Lumentum Holdings",         "LITE",  None,        "핵심공급사",  "800G/1.6T 광트랜시버 EML 레이저",             "2022~",   "Lumentum IR / LightCounting 분석"),
            ("광통신 트랜시버",              "Coherent Corp",             "COHR",  None,        "핵심공급사",  "800G/1.6T 광트랜시버 — AI 클러스터 패브릭",   "2023~",   "Coherent IR / The Register / Bloomberg"),
            ("네트워킹 ASIC",                "Marvell Technology",        "MRVL",  None,        "전략파트너",  "Ethernet/InfiniBand PHY·SERDES",              "2021~",   "Marvell IR / NVIDIA GTC 발표"),
            ("네트워킹 스위치",              "Broadcom",                  "AVGO",  None,        "공급사",      "Ethernet 스위치 Tomahawk 시리즈",             "2021~",   "Broadcom IR / The Register"),
            # ── 메모리 / 파운드리 ──
            ("HBM 메모리 (H200)",            "SK Hynix",                  None,    None,        "독점공급사",  "HBM3e — H200 GPU 핵심 메모리 독점",           "2023.10", "NVIDIA GTC 발표 / SK Hynix IR"),
            ("HBM 메모리 (B200)",            "Micron Technology",         "MU",    None,        "공급사",      "HBM3e — Blackwell B200 GPU 적용",             "2024.03", "NVIDIA GTC2024 / Micron 공시"),
            ("파운드리",                     "TSMC",                      "TSM",   None,        "독점파운드리","CoWoS 패키징 포함 4/3nm 독점 생산",            "2018~",   "TSMC IR / DigiTimes"),
            # ── AI 소프트웨어·플랫폼 ──
            ("AI 모델 허브",                 "Hugging Face",              None,    None,        "전략투자",    "오픈소스 AI 모델 허브 ($235M 라운드 참여)",   "2023.08", "Bloomberg / Hugging Face 공식발표"),
            ("AI 언어모델 (유럽)",           "Mistral AI",                None,    None,        "전략투자",    "오픈웨이트 LLM 스타트업 시리즈A 참여",        "2024.06", "Bloomberg / Les Echos"),
            ("AI 데이터 레이블링",           "Scale AI",                  None,    None,        "전략투자",    "AI 학습데이터 플랫폼 — NVIDIA GPU 수요처",    "2021~",   "Bloomberg / Scale AI 공식발표"),
            ("AI 데이터/ML플랫폼",           "Databricks",                None,    None,        "전략투자",    "엔터프라이즈 데이터·ML 플랫폼 투자 참여",     "2023~",   "Bloomberg / Databricks IR"),
            # ── GPU 클라우드·헬스케어 ──
            ("GPU 클라우드",                 "CoreWeave",                 "CRWV",  None,        "투자+고객",   "최대 GPU 클라우드 고객 & IPO 전 투자",        "2023.04", "Bloomberg / CoreWeave S-1"),
            ("AI 헬스케어",                  "Recursion Pharmaceuticals", "RXRX",  50_000_000,  "전략투자",    "AI 신약개발 플랫폼 ($50M, 2023)",             "2023.07", "SEC Form 8-K / Reuters"),
            ("AI 음성인식",                  "SoundHound AI",             "SOUN",  None,        "초기투자",    "AI 음성인식 (2024년 일부 매각)",               "2017~",   "SEC 13F (2024 Q1 공개·매각 확인)"),
            ("AI 의료영상",                  "Nano-X Imaging",            "NNOX",  10_000_000,  "전략투자",    "AI 기반 X-ray 진단 ($10M 시리즈A)",           "2020.06", "Nano-X 공시 / TechCrunch"),
            # ── 서버·인프라 ──
            ("AI 서버 ODM",                  "Super Micro Computer",      "SMCI",  None,        "핵심공급사",  "HGX·DGX AI 서버 최대 ODM — GB200 NVL72 조립", "2017~",  "SMCI IR / Bloomberg"),
            ("데이터센터 냉각",              "Vertiv",                    "VRT",   None,        "공급사",      "액냉 인프라 — GB200 NVL72 공식 파트너",       "2024.03", "NVIDIA GTC2024 / Vertiv 공시"),
        ]
    },
    "GOOGLE": {
        "display": "Alphabet (Google)",
        "ticker": "GOOGL",
        "accent": "#4285f4",
        "entries": [
            ("위성 인터넷",       "SpaceX / Starlink",    None,    900_000_000, "전략투자",    "Starlink 위성 인터넷 ($900M 시리즈C)",       "2015.01", "WSJ / SEC Alphabet 10-K"),
            ("AI 반도체 (TPU)",  "Broadcom",              "AVGO",  None,        "TPU 파트너",  "Google TPU ASIC 공동 설계 (Cloud TPU)",      "2016~",   "Google I/O 발표 / Broadcom IR"),
            ("AI 안전 플랫폼",   "Anthropic",             None,    300_000_000, "전략투자",    "AI 안전성 연구 (누적 $300M+, GCP 독점)",     "2023.02", "Bloomberg / Anthropic 공식 발표"),
            ("GPU 클라우드",     "CoreWeave",             "CRWV",  None,        "전략투자",    "GCP 연동 GPU 클라우드 파트너십",             "2024.05", "Bloomberg / CoreWeave S-1"),
            ("양자컴퓨팅",       "IonQ",                  "IONQ",  None,        "전략파트너",  "Google Cloud 양자컴퓨팅 공동연구",           "2021.10", "IonQ IPO 공시 / Google Blog"),
            ("사이버보안",       "CrowdStrike",           "CRWD",  None,        "전략파트너",  "GCP 기본 보안 파트너십 (Marketplace)",       "2023.01", "CrowdStrike IR / GCP 발표"),
            ("사이버보안 2",     "Palo Alto Networks",    "PANW",  None,        "전략파트너",  "GCP 보안 통합 (SASE / 클라우드보안)",        "2022.06", "Palo Alto IR / Google Cloud 발표"),
            ("AI 로보틱스",      "Figure AI",             None,    None,        "전략투자",    "휴머노이드 로봇 시리즈B ($675M 컨소시엄)",   "2024.02", "Bloomberg / Figure AI 공식 발표"),
            ("AI 신약개발",      "Isomorphic Labs",       None,    None,        "자회사",      "AlphaFold 기반 AI 신약설계 (Alphabet 분사)", "2021.11", "Alphabet 공식 발표 / Nature"),
            ("자율주행 (유럽)",  "Wayve",                 None,    None,        "전략투자",    "영국 자율주행 스타트업 시리즈B 투자",        "2024.05", "Bloomberg / Wayve 공식 발표"),
            ("AI 로보틱스 2",    "Apptronik",             None,    None,        "전략투자",    "휴머노이드 로봇 — NASA 출신팀 시리즈A",      "2023.10", "Bloomberg / Apptronik 공식 발표"),
            ("핀테크",           "Stripe",                None,    None,        "전략투자",    "결제 인프라 초기 투자 (GV 포트폴리오)",      "2011~",   "Crunchbase / Google Ventures 공시"),
            ("자율주행",         "Waymo",                 None,    None,        "자회사",      "로보택시 자회사 (Alphabet 100% 소유)",       "2009~",   "Alphabet 10-K / SEC 연간보고서"),
        ]
    },
    "SPACEX": {
        "display": "SpaceX (비상장)",
        "ticker": None,
        "accent": "#005288",
        "entries": [
            ("소형 발사체 경쟁사",  "Rocket Lab USA",       "RKLB",  None, "경쟁/공급망",     "소형위성 발사 & 엔진 부품 공급",              "2013~",   "RKLB IPO 공시 / SpaceNews"),
            ("위성 제조 부품",      "Redwire Corporation",  "RDW",   None, "공급사",           "위성 구조물·태양광 패널 공급",                "2020~",   "Redwire IR / SpaceNews"),
            ("위성 통신 경쟁",      "Viasat",               "VSAT",  None, "경쟁/협력",        "위성 광대역 통신 (주파수 경쟁·일부 협력)",    "2015~",   "FCC 공시 / Bloomberg"),
            ("위성 이미지·지구관측","Maxar Technologies",    None,    None, "공급사",           "고해상도 위성 이미지 — Starshield 지원",      "2021~",   "Maxar IR / SpaceNews (2023 비상장)"),
            ("방산·우주 전자",      "L3Harris Technologies","LHX",   None, "전략파트너",       "위성 통신·전자전 장비 / 미 국방부 공동수주",  "2020~",   "L3Harris IR / Defense News"),
            ("로켓 소재",           "Materion",             "MTRN",  None, "공급사",           "특수합금·Be 소재 (Merlin/Raptor 엔진)",       "2012~",   "Materion IR / Aviation Week"),
            ("데이터 분석",         "Palantir",             "PLTR",  None, "기술파트너",       "Starlink 위성 데이터 분석 플랫폼",            "2020.09", "Palantir S-1 / Reuters"),
            ("우주 인터넷 유통",    "Dish Network",         "DISH",  None, "Starlink파트너",   "Starlink 위성 인터넷 리셀러 계약",            "2021.07", "FCC 공시 / CNBC"),
            ("지상 인프라",         "Qualcomm",             "QCOM",  None, "기술파트너",       "Starlink 단말 칩셋 (스냅드래곤 기반)",        "2022.08", "Qualcomm 발표 / The Verge"),
            ("우주 보험/금융",      "AXA XL (AXA)",         None,    None, "보험파트너",       "팔콘·스타십 발사체 보험 (비상장 자회사)",     "2018~",   "AXA XL 공시 / Space Intel Report"),
        ]
    },
    "TESLA": {
        "display": "Tesla, Inc.",
        "ticker": "TSLA",
        "accent": "#e31937",
        "entries": [
            # ── 배터리 셀 ──
            ("배터리 셀 (원통형 2170/4680)", "Panasonic Holdings",      "PCRFY", None,          "합작공급사",  "기가팩토리 Nevada·일본 공동운영 — 2170·4680 전용 공급",  "2010~",   "Tesla 10-K / Panasonic IR"),
            ("배터리 셀 (LFP 중국)",          "CATL",                    None,    None,          "독점공급사",  "Model 3/Y 스탠다드 LFP 배터리 셀 (중국·유럽 물량)",     "2020~",   "Tesla 10-K / CATL 공시 / Bloomberg"),
            ("배터리 셀 (파우치)",            "LG Energy Solution",      None,    None,          "공급사",      "Model Y·S·X 파우치·원통형 셀 일부 공급",                "2018~",   "Tesla 10-K / LGES IR"),
            ("배터리 셀 (원통형 유럽)",       "Samsung SDI",             None,    None,          "공급사",      "기가팩토리 Berlin — 원통형 셀 일부 공급",               "2022~",   "Samsung SDI IR / Reuters"),
            # ── SiC 전력반도체 ──
            ("SiC 인버터 모듈",               "ON Semiconductor",        "ON",    None,          "핵심공급사",  "Model 3/Y/S/X 메인 인버터 SiC 파워모듈 독점 공급",      "2019~",   "ON Semi IR / Bloomberg / Tesla 10-K"),
            ("SiC 기판 (웨이퍼)",             "Wolfspeed",               "WOLF",  None,          "핵심공급사",  "SiC 기판 장기 공급계약 ($650M+, 2022)",                 "2022.03", "Wolfspeed SEC 8-K / Bloomberg"),
            ("SiC 칩 (유럽 생산)",            "STMicroelectronics",      "STM",   None,          "공급사",      "기가팩토리 Berlin용 SiC MOSFET 공급",                   "2021~",   "STM IR / DigiTimes"),
            # ── FSD 반도체·AI ──
            ("FSD 칩 파운드리 (HW4)",         "TSMC",                    "TSM",   None,          "독점파운드리","HW4.0 FSD·Dojo 학습칩 7nm/5nm 위탁생산",               "2019~",   "Tesla AI Day 발표 / TSMC IR"),
            ("FSD 인식 센서",                 "Mobileye (Intel)",        "MBLY",  None,          "전 공급사",   "EyeQ 칩 사용 후 2019년 자체개발로 전환",                "2014~2019","Tesla 10-K / Mobileye IPO 공시"),
            ("AI 훈련 인프라 (Dojo)",         "AMD",                     "AMD",   None,          "공급사",      "Dojo v2·HPC 서버 EPYC CPU 활용",                        "2023~",   "Tesla AI Day2 / AMD IR"),
            # ── 원자재 ──
            ("리튬 (수산화리튬)",             "Albemarle Corporation",   "ALB",   None,          "장기공급사",  "리튬 수산화물 장기 계약 (호주·칠레 광산)",              "2021~",   "Albemarle SEC 8-K / Bloomberg"),
            ("리튬 (스포듀민)",               "Piedmont Lithium",        "PLL",   None,          "공급계약",    "북미 리튬 스포듀민 공급 MOU·계약",                      "2020.09", "Piedmont SEC 8-K / Bloomberg"),
            ("리튬 (아르헨티나)",             "Arcadium Lithium",        "ALTM",  None,          "공급사",      "아르헨티나 리튬 공급 (구 Livent·Allkem 합병체)",        "2022~",   "Livent/Arcadium IR / Reuters"),
            ("희토류 자석 (모터)",            "MP Materials",            "MP",    None,          "전략파트너",  "네오디뮴 영구자석 — 구동 모터 공급",                    "2022.04", "MP Materials SEC 8-K / Bloomberg"),
            ("흑연·음극재",                   "Syrah Resources",         "SYAAF", None,          "공급계약",    "모잠비크 천연흑연 → 음극재 (기가팩토리 Nevada 공급)",   "2021~",   "Syrah IR / Tesla 공급계약 발표"),
            ("니켈 (배터리 양극재)",          "Vale S.A.",               "VALE",  None,          "공급계약",    "클래스1 니켈 장기 공급 (황산니켈)",                     "2021~",   "Vale IR / Bloomberg"),
            # ── 배터리 재활용·에너지 ──
            ("배터리 재활용",                 "Redwood Materials",       None,    None,          "전략파트너",  "폐배터리 재활용 → 양극재 재공급 (JB Straubel 설립)",    "2020~",   "Redwood Materials 공식발표 / Bloomberg"),
            ("ESS 배터리 (메가팩)",           "CATL",                    None,    None,          "ESS공급사",   "Megapack용 LFP 셀 일부 공급 (Tesla Energy)",            "2022~",   "Bloomberg / Tesla Q 공시"),
            ("충전 인프라",                   "Blink Charging",          "BLNK",  None,          "NACS파트너",  "NACS 표준 채택 — Supercharger 커넥터 공유",             "2023.06", "Blink IR / Tesla 발표"),
            ("충전 인프라 2",                 "ChargePoint Holdings",    "CHPT",  None,          "NACS파트너",  "NACS 어댑터 협력 — 북미 충전망 통합",                   "2023.06", "ChargePoint IR / SAE 표준 발표"),
            # ── 제조·부품 ──
            ("차체 알루미늄 다이캐스팅",      "IDRA Group (CPCB)",       None,    None,          "핵심공급사",  "Giga Press 6000T·9000T — 일체형 다이캐스팅",           "2020~",   "Tesla AI Day / IDRA 공식발표"),
            ("배선·와이어링 하네스",          "Aptiv PLC",               "APTV",  None,          "공급사",      "고전압 와이어링 하네스 공급",                           "2012~",   "Aptiv IR / Reuters"),
            ("유리·윈드실드",                 "AGC Inc.",                "AGCJF", None,          "공급사",      "강화유리·선루프 유리 공급 (기가팩토리 전용)",           "2019~",   "AGC IR / Nikkei"),
            # ── 자율주행 ──
            ("AI 데이터 레이블링",            "Scale AI",                None,    None,          "기술파트너",  "FSD 학습 데이터 어노테이션 협력",                       "2020~",   "Scale AI 공식발표 / Bloomberg"),
            ("로보택시·FSD",                  "Waymo (Alphabet)",        None,    None,          "경쟁사",      "레벨4 자율주행 기술 경쟁 — 데이터 수집 방식 상이",      "2018~",   "Bloomberg / Tesla AI Day"),
        ]
    },
}

def _vc_fmt_usd(v):
    if v is None: return "—"
    if v >= 1_000_000_000: return f"${v/1e9:.1f}B"
    if v >= 1_000_000:     return f"${v/1e6:.0f}M"
    return f"${v:,}"

def run_value_chain_scan():
    """벨류체인 투자 현황 스캔 — yfinance 실시간 데이터 조회"""
    print(f"\n  [벨류체인 분석] NVIDIA / Google / SpaceX / Tesla 투자 현황 조회 중...")
    results = []
    for parent, chain in VALUE_CHAIN_DB.items():
        for (area, company, ticker, inv_usd, rel_type, detail, inv_date, source) in chain["entries"]:
            row = {
                "parent":    parent,
                "display":   chain["display"],
                "area":      area,
                "company":   company,
                "ticker":    ticker or "비상장",
                "inv_usd":   inv_usd,
                "inv_fmt":   _vc_fmt_usd(inv_usd),
                "rel_type":  rel_type,
                "detail":    detail,
                "inv_date":  inv_date,
                "source":    source,
                "price":     None,
                "ret_1m":    None,
                "ret_3m":    None,
                "ret_6m":    None,
                "market_cap":None,
                "rsi":       None,
                "sector":    "",
            }
            if ticker:
                try:
                    d_df, _ = prepare_df(ticker)
                    if d_df is not None and len(d_df) >= 21:
                        cur = round(d_df["Close"].iloc[-1], 2)
                        row["price"]  = cur
                        row["ret_1m"] = round((cur / d_df["Close"].iloc[-21] - 1)*100, 1)
                        if len(d_df) >= 63:
                            row["ret_3m"] = round((cur / d_df["Close"].iloc[-63] - 1)*100, 1)
                        if len(d_df) >= 126:
                            row["ret_6m"] = round((cur / d_df["Close"].iloc[-126]- 1)*100, 1)
                        row["rsi"] = round(d_df["RSI"].iloc[-1], 1) if "RSI" in d_df.columns else None
                        # 시총 조회
                        try:
                            info = yf.Ticker(ticker).fast_info
                            mc = getattr(info, "market_cap", None)
                            if mc:
                                row["market_cap"] = _vc_fmt_usd(mc)
                        except: pass
                except: pass
            results.append(row)
    print(f"  벨류체인 분석 완료 ({len(results)}개 항목)")
    return results

def print_value_chain_results(vc_results):
    """벨류체인 결과 텍스트 출력"""
    if not vc_results:
        return
    print(f"\n{'='*70}")
    print(f"  📡 벨류체인 투자 현황 (NVIDIA / Google / SpaceX / Tesla)")
    print(f"{'='*70}")
    cur_parent = None
    for r in vc_results:
        if r["parent"] != cur_parent:
            cur_parent = r["parent"]
            print(f"\n  ── {r['display']} ──")
            print(f"  {'분야':25}  {'기업':22}  {'티커':6}  {'투자금액':8}  "
                  f"{'현재가':8}  {'1달':6}  {'3달':6}  {'관계':8}")
            print("  " + "─"*100)
        price_s = f"${r['price']:>7.2f}" if r['price'] else "   비상장"
        ret1m_s = f"{r['ret_1m']:>+5.1f}%" if r['ret_1m'] is not None else "   N/A"
        ret3m_s = f"{r['ret_3m']:>+5.1f}%" if r['ret_3m'] is not None else "   N/A"
        src_short = r.get("source","")[:28]
        print(f"  {r['area'][:25]:25}  {r['company'][:22]:22}  "
              f"{r['ticker']:>6}  {r['inv_fmt']:>8}  "
              f"{price_s}  {ret1m_s}  {ret3m_s}  "
              f"{r['rel_type']:10}  {r.get('inv_date','—'):10}  {src_short}")


# ════════════════════════════════════════════
#  ★ 세부 산업별 밸류에이션 기준 (INDUSTRY_VALUATION)
#    sector_map에 industry 필드가 있을 때 SECTOR_VALUATION 대신 우선 적용
# ════════════════════════════════════════════
INDUSTRY_VALUATION = {
    "Semiconductors":                    {"per_cheap":20,"per_fair":35,"psr_cheap":8,  "psr_fair":15, "opm_min":20},
    "Semiconductor Equipment":           {"per_cheap":18,"per_fair":30,"psr_cheap":6,  "psr_fair":12, "opm_min":20},
    "Semiconductor Equipment & Materials":{"per_cheap":18,"per_fair":30,"psr_cheap":6, "psr_fair":12, "opm_min":20},
    "Software - Infrastructure":         {"per_cheap":25,"per_fair":50,"psr_cheap":8,  "psr_fair":15, "opm_min":15},
    "Software - Application":            {"per_cheap":25,"per_fair":50,"psr_cheap":6,  "psr_fair":12, "opm_min":10},
    "Consumer Electronics":              {"per_cheap":20,"per_fair":35,"psr_cheap":5,  "psr_fair":8,  "opm_min":20},
    "IT Services":                       {"per_cheap":18,"per_fair":30,"psr_cheap":2,  "psr_fair":4,  "opm_min":12},
    "Information Technology Services":   {"per_cheap":18,"per_fair":30,"psr_cheap":2,  "psr_fair":4,  "opm_min":12},
    "Electronic Components":             {"per_cheap":15,"per_fair":25,"psr_cheap":2,  "psr_fair":4,  "opm_min":10},
    "Communication Equipment":           {"per_cheap":15,"per_fair":25,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Computer Hardware":                 {"per_cheap":15,"per_fair":25,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Scientific & Technical Instruments":{"per_cheap":20,"per_fair":35,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Internet Content & Information":    {"per_cheap":20,"per_fair":40,"psr_cheap":6,  "psr_fair":12, "opm_min":15},
    "Internet Retail":                   {"per_cheap":25,"per_fair":60,"psr_cheap":2,  "psr_fair":4,  "opm_min":5},
    "Electronic Gaming & Multimedia":    {"per_cheap":20,"per_fair":40,"psr_cheap":4,  "psr_fair":8,  "opm_min":10},
    "Biotechnology":                     {"per_cheap":20,"per_fair":50,"psr_cheap":5,  "psr_fair":15, "opm_min":5},
    "Drug Manufacturers - General":      {"per_cheap":15,"per_fair":25,"psr_cheap":4,  "psr_fair":8,  "opm_min":20},
    "Drug Manufacturers - Specialty & Generic":{"per_cheap":12,"per_fair":20,"psr_cheap":2,"psr_fair":4,"opm_min":15},
    "Medical Devices":                   {"per_cheap":20,"per_fair":35,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Medical Instruments & Supplies":    {"per_cheap":20,"per_fair":35,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Healthcare Plans":                  {"per_cheap":12,"per_fair":20,"psr_cheap":0.5,"psr_fair":1,  "opm_min":4},
    "Diagnostics & Research":            {"per_cheap":20,"per_fair":40,"psr_cheap":3,  "psr_fair":8,  "opm_min":15},
    "Medical Care Facilities":           {"per_cheap":15,"per_fair":25,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Medical Distribution":              {"per_cheap":12,"per_fair":20,"psr_cheap":0.1,"psr_fair":0.2,"opm_min":2},
    "Health Information Services":       {"per_cheap":20,"per_fair":40,"psr_cheap":4,  "psr_fair":8,  "opm_min":10},
    "Entertainment":                     {"per_cheap":18,"per_fair":35,"psr_cheap":2,  "psr_fair":5,  "opm_min":10},
    "Telecom Services":                  {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":15},
    "Advertising Agencies":              {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Broadcasting":                      {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":3,  "opm_min":10},
    "Publishing":                        {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":8},
    "Auto Manufacturers":                {"per_cheap":8, "per_fair":15,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Auto Parts":                        {"per_cheap":10,"per_fair":18,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Auto & Truck Dealerships":          {"per_cheap":8, "per_fair":14,"psr_cheap":0.1,"psr_fair":0.3,"opm_min":3},
    "Restaurants":                       {"per_cheap":20,"per_fair":35,"psr_cheap":1.5,"psr_fair":3,  "opm_min":15},
    "Specialty Retail":                  {"per_cheap":15,"per_fair":25,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Apparel Retail":                    {"per_cheap":15,"per_fair":25,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Apparel Manufacturing":             {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Footwear & Accessories":            {"per_cheap":18,"per_fair":30,"psr_cheap":1,  "psr_fair":2,  "opm_min":12},
    "Luxury Goods":                      {"per_cheap":20,"per_fair":35,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Home Improvement Retail":           {"per_cheap":18,"per_fair":28,"psr_cheap":0.8,"psr_fair":1.5,"opm_min":10},
    "Lodging":                           {"per_cheap":15,"per_fair":30,"psr_cheap":1,  "psr_fair":2,  "opm_min":15},
    "Resorts & Casinos":                 {"per_cheap":12,"per_fair":25,"psr_cheap":1,  "psr_fair":3,  "opm_min":15},
    "Leisure":                           {"per_cheap":15,"per_fair":30,"psr_cheap":1,  "psr_fair":3,  "opm_min":10},
    "Travel Services":                   {"per_cheap":15,"per_fair":30,"psr_cheap":2,  "psr_fair":5,  "opm_min":10},
    "Residential Construction":          {"per_cheap":8, "per_fair":14,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Beverages - Non-Alcoholic":         {"per_cheap":18,"per_fair":28,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Beverages - Brewers":               {"per_cheap":15,"per_fair":25,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Beverages - Alcoholic":             {"per_cheap":15,"per_fair":25,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Beverages - Wineries & Distilleries":{"per_cheap":15,"per_fair":25,"psr_cheap":2, "psr_fair":4,  "opm_min":12},
    "Discount Stores":                   {"per_cheap":20,"per_fair":35,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Grocery Stores":                    {"per_cheap":15,"per_fair":25,"psr_cheap":0.2,"psr_fair":0.5,"opm_min":3},
    "Food Distribution":                 {"per_cheap":12,"per_fair":20,"psr_cheap":0.1,"psr_fair":0.3,"opm_min":2},
    "Household & Personal Products":     {"per_cheap":18,"per_fair":28,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Packaged Foods":                    {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Confectioners":                     {"per_cheap":18,"per_fair":28,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Tobacco":                           {"per_cheap":10,"per_fair":16,"psr_cheap":1,  "psr_fair":2,  "opm_min":35},
    "Agricultural Inputs":               {"per_cheap":10,"per_fair":18,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Farm Products":                     {"per_cheap":8, "per_fair":15,"psr_cheap":0.1,"psr_fair":0.3,"opm_min":3},
    "Banks - Regional":                  {"per_cheap":8, "per_fair":14,"psr_cheap":1,  "psr_fair":3,  "opm_min":25},
    "Banks - Diversified":               {"per_cheap":9, "per_fair":14,"psr_cheap":2,  "psr_fair":4,  "opm_min":25},
    "Asset Management":                  {"per_cheap":12,"per_fair":20,"psr_cheap":2,  "psr_fair":5,  "opm_min":30},
    "Capital Markets":                   {"per_cheap":10,"per_fair":18,"psr_cheap":1,  "psr_fair":3,  "opm_min":20},
    "Insurance - Life":                  {"per_cheap":8, "per_fair":14,"psr_cheap":0.5,"psr_fair":1,  "opm_min":10},
    "Insurance - Diversified":           {"per_cheap":10,"per_fair":16,"psr_cheap":0.8,"psr_fair":1.5,"opm_min":10},
    "Insurance - Property & Casualty":   {"per_cheap":10,"per_fair":18,"psr_cheap":0.8,"psr_fair":1.5,"opm_min":10},
    "Insurance - Reinsurance":           {"per_cheap":10,"per_fair":16,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Insurance Brokers":                 {"per_cheap":18,"per_fair":30,"psr_cheap":2,  "psr_fair":4,  "opm_min":20},
    "Credit Services":                   {"per_cheap":12,"per_fair":20,"psr_cheap":2,  "psr_fair":5,  "opm_min":30},
    "Financial Data & Stock Exchanges":  {"per_cheap":20,"per_fair":40,"psr_cheap":5,  "psr_fair":12, "opm_min":30},
    "Aerospace & Defense":               {"per_cheap":18,"per_fair":30,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Specialty Industrial Machinery":    {"per_cheap":15,"per_fair":25,"psr_cheap":1.5,"psr_fair":3,  "opm_min":12},
    "Farm & Heavy Construction Machinery":{"per_cheap":12,"per_fair":20,"psr_cheap":0.8,"psr_fair":1.5,"opm_min":10},
    "Electrical Equipment & Parts":      {"per_cheap":18,"per_fair":30,"psr_cheap":2,  "psr_fair":4,  "opm_min":12},
    "Building Products & Equipment":     {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":12},
    "Building Materials":                {"per_cheap":12,"per_fair":20,"psr_cheap":0.8,"psr_fair":1.5,"opm_min":10},
    "Railroads":                         {"per_cheap":15,"per_fair":25,"psr_cheap":3,  "psr_fair":6,  "opm_min":25},
    "Air Freight & Logistics":           {"per_cheap":12,"per_fair":22,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Integrated Freight & Logistics":    {"per_cheap":12,"per_fair":22,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Airlines":                          {"per_cheap":8, "per_fair":15,"psr_cheap":0.2,"psr_fair":0.5,"opm_min":5},
    "Trucking":                          {"per_cheap":10,"per_fair":18,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Engineering & Construction":        {"per_cheap":12,"per_fair":20,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Consulting Services":               {"per_cheap":18,"per_fair":30,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Specialty Business Services":       {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Waste Management":                  {"per_cheap":20,"per_fair":35,"psr_cheap":3,  "psr_fair":5,  "opm_min":15},
    "Pollution & Treatment Controls":    {"per_cheap":15,"per_fair":25,"psr_cheap":1.5,"psr_fair":3,  "opm_min":12},
    "Industrial Distribution":           {"per_cheap":15,"per_fair":25,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":5},
    "Security & Protection Services":    {"per_cheap":18,"per_fair":30,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Conglomerates":                     {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Rental & Leasing Services":         {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":12},
    "Tools & Accessories":               {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":12},
    "Personal Services":                 {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Specialty Chemicals":               {"per_cheap":15,"per_fair":25,"psr_cheap":1.5,"psr_fair":3,  "opm_min":12},
    "Chemicals":                         {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":10},
    "Packaging & Containers":            {"per_cheap":12,"per_fair":20,"psr_cheap":0.5,"psr_fair":1,  "opm_min":8},
    "Copper":                            {"per_cheap":10,"per_fair":18,"psr_cheap":1,  "psr_fair":2,  "opm_min":15},
    "Gold":                              {"per_cheap":15,"per_fair":30,"psr_cheap":3,  "psr_fair":8,  "opm_min":20},
    "Steel":                             {"per_cheap":6, "per_fair":12,"psr_cheap":0.3,"psr_fair":0.8,"opm_min":8},
    "Other Industrial Metals & Mining":  {"per_cheap":8, "per_fair":15,"psr_cheap":0.5,"psr_fair":1,  "opm_min":10},
    "Coking Coal":                       {"per_cheap":5, "per_fair":10,"psr_cheap":0.5,"psr_fair":1,  "opm_min":15},
    "Oil & Gas E&P":                     {"per_cheap":8, "per_fair":15,"psr_cheap":1,  "psr_fair":3,  "opm_min":15},
    "Oil & Gas Integrated":              {"per_cheap":10,"per_fair":16,"psr_cheap":0.5,"psr_fair":1,  "opm_min":10},
    "Oil & Gas Midstream":               {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":15},
    "Oil & Gas Refining & Marketing":    {"per_cheap":6, "per_fair":12,"psr_cheap":0.1,"psr_fair":0.3,"opm_min":5},
    "Oil & Gas Equipment & Services":    {"per_cheap":12,"per_fair":20,"psr_cheap":0.5,"psr_fair":1,  "opm_min":10},
    "Utilities - Regulated Electric":    {"per_cheap":14,"per_fair":22,"psr_cheap":2,  "psr_fair":4,  "opm_min":15},
    "Utilities - Diversified":           {"per_cheap":14,"per_fair":22,"psr_cheap":1.5,"psr_fair":3,  "opm_min":15},
    "Utilities - Regulated Gas":         {"per_cheap":12,"per_fair":20,"psr_cheap":1,  "psr_fair":2,  "opm_min":12},
    "Utilities - Regulated Water":       {"per_cheap":18,"per_fair":28,"psr_cheap":3,  "psr_fair":6,  "opm_min":15},
    "Utilities - Renewable":             {"per_cheap":20,"per_fair":40,"psr_cheap":4,  "psr_fair":10, "opm_min":15},
    "Utilities - Independent Power Producers":{"per_cheap":15,"per_fair":25,"psr_cheap":2,"psr_fair":4,"opm_min":12},
    "Solar":                             {"per_cheap":20,"per_fair":50,"psr_cheap":3,  "psr_fair":8,  "opm_min":10},
    "REIT - Industrial":                 {"per_cheap":20,"per_fair":40,"psr_cheap":6,  "psr_fair":15, "opm_min":40},
    "REIT - Retail":                     {"per_cheap":15,"per_fair":30,"psr_cheap":4,  "psr_fair":10, "opm_min":35},
    "REIT - Residential":                {"per_cheap":18,"per_fair":35,"psr_cheap":5,  "psr_fair":12, "opm_min":35},
    "REIT - Office":                     {"per_cheap":12,"per_fair":25,"psr_cheap":3,  "psr_fair":7,  "opm_min":30},
    "REIT - Specialty":                  {"per_cheap":20,"per_fair":40,"psr_cheap":7,  "psr_fair":15, "opm_min":40},
    "REIT - Healthcare Facilities":      {"per_cheap":18,"per_fair":35,"psr_cheap":5,  "psr_fair":12, "opm_min":30},
    "REIT - Hotel & Motel":              {"per_cheap":15,"per_fair":30,"psr_cheap":3,  "psr_fair":8,  "opm_min":20},
    "REIT - Diversified":                {"per_cheap":15,"per_fair":30,"psr_cheap":4,  "psr_fair":10, "opm_min":35},
    "Real Estate Services":              {"per_cheap":15,"per_fair":25,"psr_cheap":1,  "psr_fair":3,  "opm_min":10},
}

# ── sector_map 헬퍼 함수 (구포맷 str / 신포맷 dict 모두 지원) ────────────
def get_etf_code(ticker, sector_map):
    """sector_map에서 ETF 코드 반환 — 구포맷(str)/신포맷(dict) 모두 지원"""
    val = (sector_map or {}).get(str(ticker).upper(), "")
    if isinstance(val, dict):
        return val.get("sector", "")
    return val or ""

def get_industry(ticker, sector_map):
    """sector_map에서 세부 산업명 반환 — 신포맷에서만 값 있음, 없으면 빈 문자열"""
    val = (sector_map or {}).get(str(ticker).upper(), {})
    if isinstance(val, dict):
        return val.get("industry", "")
    return ""


# ════════════════════════════════════════════
#  1. 공통 지표 계산
# ════════════════════════════════════════════
def hts_rsi(close, period=14, signal_period=9):
    delta    = close.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    rsi      = 100 - (100 / (1 + rs))
    signal   = rsi.ewm(span=signal_period, adjust=False).mean()
    return rsi, signal

def stoch_slow(high, low, close, k_period=5, slow_k=3, slow_d=3):
    lowest  = low.rolling(window=k_period).min()
    highest = high.rolling(window=k_period).max()
    fast_k  = (close - lowest) / (highest - lowest).replace(0, np.nan) * 100
    sk      = fast_k.ewm(span=slow_k, adjust=False).mean()
    sd      = sk.ewm(span=slow_d, adjust=False).mean()
    return sk, sd

def ichimoku_hts(high, low):
    tenkan = (high.rolling(9).max()  + low.rolling(9).min())  / 2
    kijun  = (high.rolling(26).max() + low.rolling(26).min()) / 2
    span_a = ((tenkan + kijun) / 2).shift(26)
    span_b = ((high.rolling(52).max() + low.rolling(52).min()) / 2).shift(26)
    return span_a, span_b

def calc_atr(df, period=14):
    high, low, close = df["High"], df["Low"], df["Close"]
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()

def prepare_df(ticker):
    d = yf.Ticker(ticker).history(period="2y")
    if len(d) < 60:
        return None, None
    d = d[["Open","High","Low","Close","Volume"]].copy()
    d.index = d.index.tz_localize(None)
    for w in [5, 20, 60, 120, 200]:
        d[f"MA{w}"] = ta.trend.SMAIndicator(d["Close"], window=w).sma_indicator()
    d["RSI"], d["RSI_sig"] = hts_rsi(d["Close"])
    macd = ta.trend.MACD(d["Close"], window_fast=12, window_slow=26, window_sign=9)
    d["MACDh"] = macd.macd_diff()
    bb = ta.volatility.BollingerBands(d["Close"], window=20, window_dev=2)
    d["BB_upper"] = bb.bollinger_hband()
    d["BB_lower"] = bb.bollinger_lband()
    d["BB_pct"]   = bb.bollinger_pband()
    d["%K"], d["%D"] = stoch_slow(d["High"], d["Low"], d["Close"])
    d["span_a"], d["span_b"] = ichimoku_hts(d["High"], d["Low"])
    d["ATR"] = calc_atr(d)
    # OBV 계산
    obv = [0]
    for i in range(1, len(d)):
        if d["Close"].iloc[i] > d["Close"].iloc[i-1]:
            obv.append(obv[-1] + d["Volume"].iloc[i])
        elif d["Close"].iloc[i] < d["Close"].iloc[i-1]:
            obv.append(obv[-1] - d["Volume"].iloc[i])
        else:
            obv.append(obv[-1])
    d["OBV"] = obv
    d["OBV_MA"] = pd.Series(obv).rolling(20).mean().values
    
    # ADX 계산
    high, low, close = d["High"], d["Low"], d["Close"]
    plus_dm  = high.diff().clip(lower=0)
    minus_dm = (-low.diff()).clip(lower=0)
    plus_dm[plus_dm < (-low.diff()).clip(lower=0)]  = 0
    minus_dm[minus_dm < high.diff().clip(lower=0)] = 0
    atr14    = calc_atr(d, period=14)
    plus_di  = 100 * (plus_dm.ewm(span=14).mean()  / atr14)
    minus_di = 100 * (minus_dm.ewm(span=14).mean() / atr14)
    dx       = (abs(plus_di - minus_di) / (plus_di + minus_di).replace(0, np.nan)) * 100
    d["ADX"]      = dx.ewm(span=14).mean()
    d["PLUS_DI"]  = plus_di
    d["MINUS_DI"] = minus_di

    w = yf.Ticker(ticker).history(period="3y", interval="1wk")
    if len(w) < 30:
        return d, None
    w = w[["Open","High","Low","Close","Volume"]].copy()
    w.index = w.index.tz_localize(None)
    for ww in [5, 10, 20, 60, 120]:
        w[f"MA{ww}"] = ta.trend.SMAIndicator(w["Close"], window=ww).sma_indicator()
    w["span_a"], w["span_b"] = ichimoku_hts(w["High"], w["Low"])
    return d, w

# ════════════════════════════════════════════
#  2. 신호 / 추세 계산
# ════════════════════════════════════════════
def weekly_trend_score(w_df):
    if w_df is None or len(w_df) < 5:
        return 0
    r = w_df.iloc[-1]
    score = 0
    if pd.notna(r.get("MA20")) and pd.notna(r.get("MA60")):
        score += 1 if r["Close"] > r["MA20"] else -1
        score += 1 if r["MA20"]  > r["MA60"]  else -1
    if pd.notna(r.get("span_a")) and pd.notna(r.get("span_b")):
        cloud_top = max(r["span_a"], r["span_b"])
        score += 1 if r["Close"] > cloud_top else -1
    return score

def count_buy_signals(d_df):
    if d_df is None or len(d_df) < 3:
        return 0, []
    r, r1 = d_df.iloc[-1], d_df.iloc[-2]
    sigs = []

    if pd.notna(r.get("span_a")) and pd.notna(r.get("span_b")):
        if r["Close"] > max(r["span_a"], r["span_b"]):
            sigs.append("구름대 위")

    if pd.notna(r.get("MA5")) and pd.notna(r.get("MA20")) and pd.notna(r.get("MA60")):
        if r["MA5"] > r["MA20"] > r["MA60"]:
            sigs.append("MA 정배열")

    if pd.notna(r.get("RSI")) and pd.notna(r.get("RSI_sig")):
        if r1["RSI"] <= r1["RSI_sig"] and r["RSI"] > r["RSI_sig"] and r["RSI"] < 65:
            sigs.append("RSI 상향돌파")

    if pd.notna(r.get("MACDh")) and pd.notna(r1.get("MACDh")):
        if r1["MACDh"] < 0 and r["MACDh"] >= 0:
            sigs.append("MACD 양전환")

    if pd.notna(r.get("%K")) and pd.notna(r.get("%D")):
        if r["%K"] < 35 and r1["%K"] <= r1["%D"] and r["%K"] > r["%D"]:
            sigs.append(f"Stoch GC(K={r['%K']:.0f})")

    if pd.notna(r.get("BB_pct")) and r["BB_pct"] < 0.25:
        sigs.append(f"BB하단({r['BB_pct']:.2f})")

    vol_ma = d_df["Volume"].rolling(20).mean().iloc[-1]
    if pd.notna(vol_ma) and r["Volume"] > vol_ma * 1.5 and r["Close"] >= r["Open"]:
        sigs.append(f"거래량급증({r['Volume']/vol_ma:.1f}x)")

    # RSI 상승 기울기 — 최근 3일 RSI 우상향 + 30~70 구간
    rsi_3d_ago = d_df["RSI"].iloc[-3]
    if pd.notna(r.get("RSI")) and pd.notna(rsi_3d_ago):
        rsi_slope = r["RSI"] - rsi_3d_ago
        if rsi_slope > 3 and 30 < r["RSI"] < 70:
            sigs.append(f"RSI상승기울기(+{rsi_slope:.1f})")

    # OBV 상승 — 보정 점수만
    if "OBV" in d_df.columns and "OBV_MA" in d_df.columns:
        if pd.notna(r.get("OBV")) and pd.notna(r.get("OBV_MA")):
            if r["OBV"] > r["OBV_MA"] and d_df["OBV"].iloc[-1] > d_df["OBV"].iloc[-5]:
                sigs.append(f"OBV상승({r['OBV']/r['OBV_MA']:.2f}x)")
    
    # ADX 강세 — 보정 점수만
    if "ADX" in d_df.columns and pd.notna(r.get("ADX")):
        if r["ADX"] > 25:
            sigs.append(f"ADX강세({r['ADX']:.1f})")

    return len(sigs), sigs

def count_sell_signals(d_df):
    """매도 신호 카운트"""
    if d_df is None or len(d_df) < 2:
        return 0, []
    r, r1 = d_df.iloc[-1], d_df.iloc[-2]
    sigs = []

    # 가격이 구름대 아래
    if pd.notna(r.get("span_a")) and pd.notna(r.get("span_b")):
        if r["Close"] < min(r["span_a"], r["span_b"]):
            sigs.append("구름대 아래")

    # MA 역배열
    if pd.notna(r.get("MA5")) and pd.notna(r.get("MA20")) and pd.notna(r.get("MA60")):
        if r["MA5"] < r["MA20"] < r["MA60"]:
            sigs.append("MA 역배열")

    # RSI Signal 하향 돌파
    if pd.notna(r.get("RSI")) and pd.notna(r.get("RSI_sig")):
        if r1["RSI"] >= r1["RSI_sig"] and r["RSI"] < r["RSI_sig"] and r["RSI"] > 35:
            sigs.append(f"RSI 하향돌파({r['RSI']:.1f})")

    # MACD 히스토그램 음전환
    if pd.notna(r.get("MACDh")) and pd.notna(r1.get("MACDh")):
        if r1["MACDh"] > 0 and r["MACDh"] <= 0:
            sigs.append("MACD 음전환")

    # Stoch 데드크로스 (65 이상)
    if pd.notna(r.get("%K")) and pd.notna(r.get("%D")):
        if r["%K"] > 65 and r1["%K"] >= r1["%D"] and r["%K"] < r["%D"]:
            sigs.append(f"Stoch DC(K={r['%K']:.0f})")

    # BB 상단 이탈
    if pd.notna(r.get("BB_pct")) and r["BB_pct"] > 0.90:
        sigs.append(f"BB상단({r['BB_pct']:.2f})")

    # 거래량 급증 + 음봉
    vol_ma = d_df["Volume"].rolling(20).mean().iloc[-1]
    if pd.notna(vol_ma) and r["Volume"] > vol_ma * 1.5 and r["Close"] < r["Open"]:
        sigs.append(f"거래량급증+음봉({r['Volume']/vol_ma:.1f}x)")

    return len(sigs), sigs

# ════════════════════════════════════════════
#  공통 유틸 — 시장 이벤트 감지
# ════════════════════════════════════════════

def get_triple_witching_info():
    """
    세 마녀의 날 정보 반환
    → 3/6/9/12월 세 번째 금요일
    """
    from datetime import date, timedelta
    
    today = date.today()
    
    def third_friday(year, month):
        # 해당 월 첫 번째 날
        d = date(year, month, 1)
        # 첫 번째 금요일 찾기
        days_until_friday = (4 - d.weekday()) % 7
        first_friday = d + timedelta(days=days_until_friday)
        # 세 번째 금요일
        return first_friday + timedelta(weeks=2)
    
    # 세 마녀의 달: 3, 6, 9, 12월
    tw_months = [3, 6, 9, 12]
    
    # 올해/내년 세 마녀의 날 목록
    tw_dates = []
    for year in [today.year, today.year + 1]:
        for month in tw_months:
            tw_dates.append(third_friday(year, month))
    
    # 오늘 이후 가장 가까운 세 마녀의 날
    future_tw = [d for d in tw_dates if d >= today]
    next_tw   = min(future_tw) if future_tw else None
    
    if next_tw is None:
        return {"days_until": 999, "is_today": False, "date": None, "warning": False}
    
    days_until = (next_tw - today).days
    is_today   = days_until == 0
    warning    = days_until <= 7  # 7일 이내 경고
    
    return {
        "days_until": days_until,
        "is_today":   is_today,
        "date":       next_tw.strftime("%Y-%m-%d"),
        "warning":    warning,
        "label":      "🚨 세 마녀의 날!" if is_today else
                      f"⚠ 세 마녀의 날 {days_until}일 전" if warning else
                      f"세 마녀의 날 {days_until}일 후",
    }

def get_vix_info():
    """
    VIX 지수 기반 시장 심리 판단
    반환: dict
    """
    try:
        vix_df, _ = prepare_df("^VIX")
        if vix_df is None or vix_df.empty:
            return {"vix": None, "label": "N/A", "status": "N/A", "warning": False}
        
        vix = round(vix_df["Close"].iloc[-1], 2)
        vix_ma20 = round(vix_df["Close"].rolling(20).mean().iloc[-1], 2)
        vix_1w_ago = round(vix_df["Close"].iloc[-5], 2) if len(vix_df) >= 5 else vix
        vix_trend = "상승" if vix > vix_1w_ago else "하락"
        
        # VIX 레벨 판정
        if vix >= 40:
            status  = "🚨 극도의 공포"
            warning = True
            signal  = "역발상 매수 기회 (극단적 공포)"
            color   = "#dc2626"
        elif vix >= 30:
            status  = "🔴 공포 구간"
            warning = True
            signal  = "분할매수 고려 (공포 구간)"
            color   = "#dc2626"
        elif vix >= 20:
            status  = "🟡 경계 구간"
            warning = True
            signal  = "신규 진입 주의 (변동성 확대)"
            color   = "#d97706"
        elif vix >= 15:
            status  = "🟢 정상 구간"
            warning = False
            signal  = "정상 시장 환경"
            color   = "#16a34a"
        else:
            status  = "🔵 과열 구간"
            warning = False
            signal  = "시장 과열 주의 (VIX 과도하게 낮음)"
            color   = "#3b82f6"
        
        return {
            "vix":      vix,
            "vix_ma20": vix_ma20,
            "vix_trend": vix_trend,
            "status":   status,
            "signal":   signal,
            "warning":  warning,
            "color":    color,
            "label":    f"VIX {vix} ({status})",
        }
    except Exception as e:
        return {"vix": None, "label": "N/A", "status": "N/A", 
                "warning": False, "signal": "", "color": "#9ca3af"}

def get_earnings_info(ticker):
    """
    어닝 발표일 + 서프라이즈 정보
    반환: dict
    """
    try:
        t = yf.Ticker(ticker)
        
        # 다음 어닝 발표일
        cal = t.calendar
        next_earnings = None
        if cal is not None and not cal.empty:
            if "Earnings Date" in cal.index:
                ed = cal.loc["Earnings Date"]
                if hasattr(ed, '__iter__'):
                    next_earnings = pd.to_datetime(ed.iloc[0]).date()
                else:
                    next_earnings = pd.to_datetime(ed).date()
        
        # 어닝 히스토리 (서프라이즈)
        eh = t.earnings_history
        surprise_pct = None
        last_surprise = None
        days_since_earnings = None
        
        if eh is not None and not eh.empty:
            latest = eh.iloc[0]
            eps_est    = latest.get("epsEstimate", None)
            eps_actual = latest.get("epsActual",   None)
            
            if eps_est and eps_actual and eps_est != 0:
                surprise_pct = round((eps_actual - eps_est) / abs(eps_est) * 100, 1)
            
            # 발표일로부터 경과일
            if "quarter" in latest.index or hasattr(latest, "name"):
                try:
                    earn_date = pd.to_datetime(eh.index[0]).date()
                    days_since_earnings = (date.today() - earn_date).days
                except:
                    pass
        
        # 다음 발표일까지 잔여일
        from datetime import date
        days_until_earnings = None
        if next_earnings:
            days_until_earnings = (next_earnings - date.today()).days
        
        # 경고 레벨
        warning = ""
        if days_until_earnings is not None:
            if days_until_earnings <= 3:
                warning = "🚨 어닝 3일 이내"
            elif days_until_earnings <= 7:
                warning = "⚠ 어닝 7일 이내"
        
        # 서프라이즈 레이블
        surprise_label = ""
        if surprise_pct is not None:
            if surprise_pct >= 10:
                surprise_label = f"🔥 +{surprise_pct}% 서프라이즈"
            elif surprise_pct >= 5:
                surprise_label = f"⭐ +{surprise_pct}% 서프라이즈"
            elif surprise_pct >= 0:
                surprise_label = f"✅ +{surprise_pct}%"
            else:
                surprise_label = f"🔴 {surprise_pct}% 미스"
        
        return {
            "next_earnings":        str(next_earnings) if next_earnings else "N/A",
            "days_until_earnings":  days_until_earnings,
            "surprise_pct":         surprise_pct,
            "surprise_label":       surprise_label,
            "days_since_earnings":  days_since_earnings,
            "warning":              warning,
        }
    except Exception as e:
        return {
            "next_earnings": "N/A",
            "days_until_earnings": None,
            "surprise_pct": None,
            "surprise_label": "N/A",
            "days_since_earnings": None,
            "warning": "",
        }
    
def get_option_iv(ticker):
    """
    옵션 IV (Implied Volatility) 계산
    ATM 옵션 기준 IV 반환
    """
    try:
        t = yf.Ticker(ticker)
        
        # 현재가
        current_price = t.fast_info.last_price
        if not current_price:
            return {"iv": None, "pcr": None, "label": "N/A", "warning": False}
        
        # 옵션 만기일 목록
        expirations = t.options
        if not expirations:
            return {"iv": None, "pcr": None, "label": "N/A", "warning": False}
        
        # 가장 가까운 만기일 사용
        exp = expirations[0]
        opt_chain = t.option_chain(exp)
        
        calls = opt_chain.calls
        puts  = opt_chain.puts
        
        if calls.empty or puts.empty:
            return {"iv": None, "pcr": None, "label": "N/A", "warning": False}
        
        # ATM 콜 옵션 IV (현재가에 가장 가까운 행사가)
        calls["dist"] = abs(calls["strike"] - current_price)
        atm_call = calls.loc[calls["dist"].idxmin()]
        iv = atm_call.get("impliedVolatility", None)
        
        # Put/Call Ratio (거래량 기준)
        call_vol = calls["volume"].sum()
        put_vol  = puts["volume"].sum()
        pcr = round(put_vol / call_vol, 2) if call_vol > 0 else None
        
        # IV 레벨 판정
        if iv is None:
            iv_label  = "N/A"
            iv_warning = False
        else:
            iv_pct = round(iv * 100, 1)
            if iv_pct > 80:
                iv_label   = f"🚨 IV {iv_pct}% (극고변동성)"
                iv_warning = True
            elif iv_pct > 50:
                iv_label   = f"⚠ IV {iv_pct}% (고변동성)"
                iv_warning = True
            elif iv_pct > 30:
                iv_label   = f"IV {iv_pct}% (보통)"
                iv_warning = False
            else:
                iv_label   = f"IV {iv_pct}% (저변동성)"
                iv_warning = False
        
        # PCR 판정
        if pcr is not None:
            if pcr > 1.0:
                pcr_label = f"PCR {pcr} (비관적)"
            elif pcr < 0.5:
                pcr_label = f"PCR {pcr} (낙관적/과열주의)"
            else:
                pcr_label = f"PCR {pcr} (중립)"
        else:
            pcr_label = "PCR N/A"
        
        return {
            "iv":        round(iv * 100, 1) if iv else None,
            "iv_label":  iv_label,
            "pcr":       pcr,
            "pcr_label": pcr_label,
            "warning":   iv_warning,
            "exp_date":  exp,
        }
    except Exception as e:
        return {"iv": None, "pcr": None, "iv_label": "N/A",
                "pcr_label": "N/A", "warning": False, "exp_date": "N/A"}

def check_fundamental(ticker, etf_code=""):
    """
    대차대조표 기반 펀더멘털 체크리스트 (8가지)
    섹터별 예외 처리 포함
    반환: (위험_개수, 위험_항목_리스트, 판정)
    """
    # 섹터별 예외 항목 정의
    SECTOR_EXCEPTIONS = {
        "XLF":  [5, 6, 7],   # 금융: 무형자산/기타장기자산/우선주 구조적 정상
        "XLRE": [5, 6],       # 부동산: 무형자산/기타장기자산 정상
        "XLK":  [5],          # 기술: 무형자산 높음 정상 (소프트웨어 특성)
        "XLV":  [5],          # 헬스케어: 무형자산 높음 정상 (특허/IP)
        "XLI":  [2],          # 산업재: 매출채권 증가 일부 정상 (B2B)
        "XLU":  [6],          # 유틸리티: 기타장기자산 높음 정상 (인프라)
    }
    exceptions = SECTOR_EXCEPTIONS.get(etf_code, [])

    try:
        ticker_obj = yf.Ticker(ticker)
        bs = ticker_obj.balance_sheet      # 대차대조표
        fs = ticker_obj.financials         # 손익계산서
        cf = ticker_obj.cashflow           # 현금흐름표

        if bs is None or bs.empty:
            return 0, [], "데이터없음"

        # 최신 / 전년도 컬럼
        col0 = bs.columns[0]   # 최신
        col1 = bs.columns[1] if len(bs.columns) > 1 else col0

        def get(df, *keys):
            """여러 키 중 존재하는 첫 번째 값 반환"""
            for k in keys:
                if k in df.index:
                    v = df.loc[k, col0]
                    return float(v) if pd.notna(v) else 0.0
            return 0.0

        def get_prev(df, *keys):
            for k in keys:
                if k in df.index:
                    v = df.loc[k, col1]
                    return float(v) if pd.notna(v) else 0.0
            return 0.0

        # 주요 항목 추출
        cash          = get(bs, "Cash And Cash Equivalents",
                             "Cash Cash Equivalents And Short Term Investments")
        total_debt    = get(bs, "Total Debt", "Long Term Debt And Capital Lease Obligation")
        current_debt  = get(bs, "Current Debt", "Current Debt And Capital Lease Obligation",
                             "Short Long Term Debt")
        total_assets  = get(bs, "Total Assets")
        intangibles   = get(bs, "Goodwill And Other Intangible Assets",
                             "Net PPE", "Goodwill")
        other_assets  = get(bs, "Other Non Current Assets", "Other Assets")
        preferred     = get(bs, "Preferred Stock", "Preferred Securities Outside StockHolders Equity")
        equity        = get(bs, "Stockholders Equity", "Common Stock Equity")
        inventory     = get(bs, "Inventory")
        receivables   = get(bs, "Receivables", "Accounts Receivable")

        # 전년도
        inventory_prev    = get_prev(bs, "Inventory")
        receivables_prev  = get_prev(bs, "Receivables", "Accounts Receivable")

        # 손익계산서
        revenue     = 0.0
        revenue_prev= 0.0
        net_income  = 0.0
        if fs is not None and not fs.empty:
            fc0 = fs.columns[0]
            fc1 = fs.columns[1] if len(fs.columns) > 1 else fc0
            for k in ["Total Revenue", "Operating Revenue"]:
                if k in fs.index:
                    revenue      = float(fs.loc[k, fc0]) if pd.notna(fs.loc[k, fc0]) else 0.0
                    revenue_prev = float(fs.loc[k, fc1]) if pd.notna(fs.loc[k, fc1]) else 0.0
                    break
            for k in ["Net Income", "Net Income Common Stockholders"]:
                if k in fs.index:
                    net_income = float(fs.loc[k, fc0]) if pd.notna(fs.loc[k, fc0]) else 0.0
                    break

        # ── 체크리스트 판정 ──────────────────────
        risks = []

        # 1. 현금 < 총부채 → 유동성 위기
        if 1 not in exceptions:
            if total_debt > 0 and cash < total_debt:
                risks.append("현금<총부채")

        # 2. 매출채권 증가율 > 매출 증가율 → 외상 매출 위험
        if 2 not in exceptions:
            if revenue_prev > 0 and receivables_prev > 0:
                rev_growth = (revenue - revenue_prev) / abs(revenue_prev)
                rec_growth = (receivables - receivables_prev) / abs(receivables_prev)
                if rec_growth > rev_growth + 0.05:   # 5%p 이상 차이
                    risks.append("매출채권↑>매출↑")

        # 3. 재고 증가율 > 순이익 증가율 → 수요 감소
        if 3 not in exceptions:
            if inventory_prev > 0 and inventory > inventory_prev:
                inv_growth = (inventory - inventory_prev) / abs(inventory_prev)
                if net_income < 0 or inv_growth > 0.15:   # 재고 15% 이상 증가
                    risks.append("재고↑이익↓")

        # 4. 단기부채 > 보유현금 → 채무 상환 능력 부족
        if 4 not in exceptions:
            if current_debt > 0 and current_debt > cash:
                risks.append("단기부채>현금")

        # 5. 무형자산 > 총자산 50% → 자산가치 변동성
        if 5 not in exceptions:
            if total_assets > 0 and intangibles / total_assets > 0.5:
                risks.append(f"무형자산{intangibles/total_assets*100:.0f}%")

        # 6. 기타장기자산 > 총자산 50% → 불투명한 자산
        if 6 not in exceptions:
            if total_assets > 0 and other_assets / total_assets > 0.5:
                risks.append(f"기타자산{other_assets/total_assets*100:.0f}%")

        # 7. 우선주 존재 → 일반 주주 불리
        if 7 not in exceptions:
            if preferred > 0:
                risks.append("우선주존재")

        # 8. 자기자본 음수 → 자본잠식
        if 8 not in exceptions:
            if equity < 0:
                risks.append("자본잠식")

        # 판정
        risk_count = len(risks)
        if risk_count == 0:
            judgment = "✅ 양호"
        elif risk_count <= 2:
            judgment = "⚠ 주의"
        else:
            judgment = "🔴 위험"

        return risk_count, risks, judgment

    except Exception as e:
        return 0, [], "조회실패"
    
def check_growth_score(ticker, etf_code=""):
    """
    성장주 스코어 계산
    반환: (score, details_dict)
    score: 0~10점
    """
        # 섹터별 밸류에이션 기준
    SECTOR_VALUATION = {
        "XLK":  {"per_cheap": 25, "per_fair": 40, "psr_cheap": 5,  "psr_fair": 10, "opm_min": 15},
        "XLC":  {"per_cheap": 20, "per_fair": 35, "psr_cheap": 3,  "psr_fair": 6,  "opm_min": 15},
        "XLY":  {"per_cheap": 18, "per_fair": 30, "psr_cheap": 1,  "psr_fair": 2,  "opm_min": 8},
        "XLP":  {"per_cheap": 18, "per_fair": 25, "psr_cheap": 1,  "psr_fair": 2,  "opm_min": 10},
        "XLF":  {"per_cheap": 10, "per_fair": 15, "psr_cheap": 2,  "psr_fair": 4,  "opm_min": 20},
        "XLV":  {"per_cheap": 18, "per_fair": 30, "psr_cheap": 3,  "psr_fair": 6,  "opm_min": 10},
        "XLI":  {"per_cheap": 15, "per_fair": 25, "psr_cheap": 1,  "psr_fair": 2,  "opm_min": 10},
        "XLB":  {"per_cheap": 12, "per_fair": 20, "psr_cheap": 1,  "psr_fair": 2,  "opm_min": 10},
        "XLE":  {"per_cheap": 10, "per_fair": 15, "psr_cheap": 0.5,"psr_fair": 1,  "opm_min": 15},
        "XLU":  {"per_cheap": 14, "per_fair": 20, "psr_cheap": 2,  "psr_fair": 3,  "opm_min": 15},
        "XLRE": {"per_cheap": 20, "per_fair": 35, "psr_cheap": 3,  "psr_fair": 6,  "opm_min": 20},
    }
    # 섹터 기준 로드 (없으면 기본값)
    try:
        info = yf.Ticker(ticker).info
        # 세부산업 기준 우선, 없으면 섹터 기준 폴백
        _ind = info.get("industry", "") if isinstance(info, dict) else ""
        v = INDUSTRY_VALUATION.get(_ind) or SECTOR_VALUATION.get(etf_code, {
            "per_cheap": 15, "per_fair": 25,
            "psr_cheap": 2,  "psr_fair": 5,
            "opm_min": 10
        })
        fs   = yf.Ticker(ticker).financials

        score   = 0
        details = {}

        # ── 1) 시가총액 필터 ──────────────────
        market_cap = info.get("marketCap", 0)
        if market_cap > 0:
            if market_cap < 10_000_000_000:      # $10B 미만 소형주
                score += 3
                details["시총"] = f"소형 ${market_cap/1e9:.1f}B ⭐⭐⭐"
            elif market_cap < 100_000_000_000:    # $100B 미만 중형주
                score += 1
                details["시총"] = f"중형 ${market_cap/1e9:.1f}B ⭐"
            else:
                details["시총"] = f"대형 ${market_cap/1e9:.1f}B"

        # ── 2) PEG 비율 ───────────────────────
        peg = info.get("pegRatio")
        if peg and peg > 0:
            if peg < 1.0:
                score += 3
                details["PEG"] = f"{peg:.2f} ⭐⭐⭐ (성장대비 저평가)"
            elif peg < 2.0:
                score += 1
                details["PEG"] = f"{peg:.2f} ⭐ (적정)"
            else:
                details["PEG"] = f"{peg:.2f} (고평가)"
        else:
            details["PEG"] = "N/A"

        # ── 3) PSR (Price/Sales) ──────────────
        psr = info.get("priceToSalesTrailing12Months")
        if psr and psr > 0:
            if psr < v["psr_cheap"]:
                score += 2
                details["PSR"] = f"{psr:.2f} ⭐⭐ (섹터 저평가)"
            elif psr < v["psr_fair"]:
                score += 1
                details["PSR"] = f"{psr:.2f} ⭐ (섹터 적정)"
            else:
                details["PSR"] = f"{psr:.2f} (섹터 고평가)"
        else:
            details["PSR"] = "N/A"

        # PER 섹터 기준 반영
        per = info.get("trailingPE")
        if per and per > 0:
            if per < v["per_cheap"]:
                score += 2
                details["PER"] = f"{per:.1f} ⭐⭐ (섹터 저평가)"
            elif per < v["per_fair"]:
                score += 1
                details["PER"] = f"{per:.1f} ⭐ (섹터 적정)"
            else:
                details["PER"] = f"{per:.1f} (섹터 고평가)"
        else:
            details["PER"] = "N/A"

        # Forward PER 섹터 기준 반영
        fwd_per = info.get("forwardPE")
        if fwd_per and fwd_per > 0:
            if fwd_per < v["per_cheap"]:
                score += 2
                details["ForwardPER"] = f"{fwd_per:.1f} ⭐⭐ (섹터 저평가)"
            elif fwd_per < v["per_fair"]:
                score += 1
                details["ForwardPER"] = f"{fwd_per:.1f} ⭐ (섹터 적정)"
            else:
                details["ForwardPER"] = f"{fwd_per:.1f} (섹터 고평가)"
            
            # Trailing vs Forward 비교 → 이익 개선 여부
            per = info.get("trailingPE")
            if per and per > 0 and fwd_per < per:
                score += 1
                details["PER개선"] = f"Trailing {per:.1f} → Forward {fwd_per:.1f} ⭐ (이익개선)"
        else:
            details["ForwardPER"] = "N/A"

        # ── 4) EPS 성장률 ─────────────────────
        if fs is not None and not fs.empty and len(fs.columns) >= 2:
            fc0, fc1 = fs.columns[0], fs.columns[1]
            net0 = net1 = 0
            for k in ["Net Income", "Net Income Common Stockholders"]:
                if k in fs.index:
                    net0 = float(fs.loc[k, fc0]) if pd.notna(fs.loc[k, fc0]) else 0
                    net1 = float(fs.loc[k, fc1]) if pd.notna(fs.loc[k, fc1]) else 0
                    break
            if net1 != 0 and net0 > 0:
                eps_growth = (net0 - net1) / abs(net1) * 100
                if eps_growth > 30:
                    score += 3
                    details["EPS성장"] = f"+{eps_growth:.1f}% ⭐⭐⭐"
                elif eps_growth > 10:
                    score += 1
                    details["EPS성장"] = f"+{eps_growth:.1f}% ⭐"
                else:
                    details["EPS성장"] = f"{eps_growth:.1f}%"
            else:
                details["EPS성장"] = "N/A"

        # ── 5) OPM (영업이익률) 개선 ──────────
        if fs is not None and not fs.empty and len(fs.columns) >= 2:
            fc0, fc1 = fs.columns[0], fs.columns[1]
            op0 = op1 = rev0 = rev1 = 0
            for k in ["Operating Income", "EBIT"]:
                if k in fs.index:
                    op0 = float(fs.loc[k, fc0]) if pd.notna(fs.loc[k, fc0]) else 0
                    op1 = float(fs.loc[k, fc1]) if pd.notna(fs.loc[k, fc1]) else 0
                    break
            for k in ["Total Revenue", "Operating Revenue"]:
                if k in fs.index:
                    rev0 = float(fs.loc[k, fc0]) if pd.notna(fs.loc[k, fc0]) else 0
                    rev1 = float(fs.loc[k, fc1]) if pd.notna(fs.loc[k, fc1]) else 0
                    break
            if rev0 > 0 and rev1 > 0:
                opm0 = op0 / rev0 * 100
                opm1 = op1 / rev1 * 100
                opm_delta = opm0 - opm1
                if opm0 >= v["opm_min"] and opm_delta > 3:      # ← 교체
                    score += 2
                    details["OPM"] = f"{opm0:.1f}% (▲{opm_delta:.1f}%p) ⭐⭐"
                elif opm0 >= v["opm_min"] and opm_delta > 0:    # ← 교체
                    score += 1
                    details["OPM"] = f"{opm0:.1f}% (▲{opm_delta:.1f}%p) ⭐"
                elif opm_delta > 3:                              # ← 교체
                    score += 1
                    details["OPM"] = f"{opm0:.1f}% (▲{opm_delta:.1f}%p) ⭐ (개선중)"
                else:
                    details["OPM"] = f"{opm0:.1f}% (▼{abs(opm_delta):.1f}%p)"
            else:
                details["OPM"] = "N/A"

        # ── 6) 애널리스트 목표주가 상향 ───────
        target_mean  = info.get("targetMeanPrice", 0)
        current      = info.get("currentPrice", 0)
        num_analysts = info.get("numberOfAnalystOpinions", 0)
        if target_mean and current and num_analysts >= 3:
            upside = (target_mean / current - 1) * 100
            if upside > 20:
                score += 2
                details["목표주가"] = f"${target_mean:.0f} (▲{upside:.1f}%) ⭐⭐"
            elif upside > 10:
                score += 1
                details["목표주가"] = f"${target_mean:.0f} (▲{upside:.1f}%) ⭐"
            else:
                details["목표주가"] = f"${target_mean:.0f} (▲{upside:.1f}%)"
        else:
            details["목표주가"] = "N/A"

        # ── 7) 내부자 매수 ────────────────────
        try:
            insider = yf.Ticker(ticker).insider_purchases
            if insider is not None and not insider.empty:
                # 최근 3개월 내 순매수 여부
                recent = insider.head(5)
                net_buy = recent.get("Shares", pd.Series()).sum()
                if net_buy > 0:
                    score += 1
                    details["내부자"] = f"순매수 ⭐"
                else:
                    details["내부자"] = "중립/매도"
            else:
                details["내부자"] = "N/A"
        except:
            details["내부자"] = "N/A"

        # ── 8) 어닝 서프라이즈 ───────────────
        earn_info = get_earnings_info(ticker)
        
        # 어닝 발표 임박 경고 (감점)
        if earn_info["days_until_earnings"] is not None:
            if earn_info["days_until_earnings"] <= 3:
                score -= 1
                details["어닝"] = f"🚨 {earn_info['days_until_earnings']}일 후 발표 (진입 주의)"
            elif earn_info["days_until_earnings"] <= 7:
                details["어닝"] = f"⚠ {earn_info['days_until_earnings']}일 후 발표"
            else:
                details["어닝"] = f"발표 {earn_info['days_until_earnings']}일 후"
        
        # 어닝 서프라이즈 반영
        if earn_info["surprise_pct"] is not None:
            sp = earn_info["surprise_pct"]
            ds = earn_info["days_since_earnings"] or 999
            
            if sp >= 10 and 4 <= ds <= 10:
                # 긍정 서프라이즈 + 발표 후 4~10일 (적절한 진입 타이밍)
                score += 2
                details["서프라이즈"] = f"🔥 +{sp}% ({ds}일 전) ⭐⭐"
            elif sp >= 5 and 4 <= ds <= 10:
                score += 1
                details["서프라이즈"] = f"⭐ +{sp}% ({ds}일 전)"
            elif sp >= 10 and ds <= 3:
                # 긍정 서프라이즈지만 너무 직후 → 추격 주의
                details["서프라이즈"] = f"🔥 +{sp}% ({ds}일 전) 추격 주의"
            elif sp < 0:
                score -= 1
                details["서프라이즈"] = f"🔴 {sp}% 미스"
            else:
                details["서프라이즈"] = f"{earn_info['surprise_label']}"

        # ── 9) 옵션 IV ────────────────────────
        iv_info = get_option_iv(ticker)
        if iv_info["iv"] is not None:
            if iv_info["iv"] > 80:
                score -= 1  # 극고변동성 → 진입 위험
                details["IV"] = iv_info["iv_label"]
            elif iv_info["iv"] > 50:
                details["IV"] = iv_info["iv_label"]
            else:
                details["IV"] = iv_info["iv_label"]
        else:
            details["IV"] = "N/A"
        
        if iv_info["pcr"] is not None:
            details["PCR"] = iv_info["pcr_label"]

        # ── 판정 ──────────────────────────────
        if score >= 8:
            grade = "🔥 최우선"
        elif score >= 5:
            grade = "⭐ 유망"
        elif score >= 3:
            grade = "✅ 관심"
        else:
            grade = "➖ 보통"

        return score, grade, details

    except Exception as e:
        return 0, "조회실패", {}

def calc_stop_target(d_df, entry):
    if d_df is None or len(d_df) < 20:
        return entry * 0.95, entry * 1.10, 0
    r = d_df.iloc[-1]
    atr = r["ATR"] if pd.notna(r.get("ATR")) else entry * 0.03
    recent_low = d_df["Low"].tail(20).min()
    cloud_bot  = min(r["span_a"], r["span_b"]) if pd.notna(r.get("span_a")) else recent_low
    stop_swing = max(recent_low, cloud_bot) * 0.995
    stop_atr   = entry - atr * 2.0
    stop       = max(stop_swing, stop_atr)
    risk       = entry - stop
    target_rr  = entry + risk * 2.0
    target_bb  = r["BB_upper"] if pd.notna(r.get("BB_upper")) else entry * 1.10
    target     = float(np.median([target_rr, target_bb, target_rr]))
    rr         = (target - entry) / risk if risk > 0 else 0

    # 피보나치 되돌림 레벨 계산
    high_recent = d_df["High"].tail(60).max()
    low_recent  = d_df["Low"].tail(60).min()
    fib_382 = round(high_recent - (high_recent - low_recent) * 0.382, 2)
    fib_618 = round(high_recent - (high_recent - low_recent) * 0.618, 2)

    return round(stop, 2), round(target, 2), round(rr, 2)

def generate_chart_base64(ticker, d_df):
    """일봉 180일 차트 생성 → Base64 문자열 반환"""
    import io, base64
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from matplotlib.patches import Rectangle

    # 한글 폰트
    import platform
    _sys = platform.system()
    font = "Malgun Gothic" if _sys=="Windows" else \
           "AppleGothic"   if _sys=="Darwin"  else "NanumGothic"
    mpl.rcParams["font.family"]        = font
    mpl.rcParams["axes.unicode_minus"] = False

    df = d_df.tail(180).copy()
    if len(df) < 30:
        return None

    fig = plt.figure(figsize=(14, 12))
    fig.suptitle(f"{ticker} — 일봉 180일", fontsize=13, fontweight="bold")
    gs = gridspec.GridSpec(4, 1, figure=fig,
                           height_ratios=[3, 1, 1, 1],
                           hspace=0.08)

    # ── 1. 캔들 + MA + BB + 일목 ──────────────
    ax1 = fig.add_subplot(gs[0])
    for i, (idx, row) in enumerate(df.iterrows()):
        color = "#ef4444" if row["Close"] >= row["Open"] else "#3b82f6"
        ax1.plot([i, i], [row["Low"], row["High"]], color=color, lw=0.8)
        ax1.add_patch(Rectangle(
            (i-0.3, min(row["Open"], row["Close"])),
            0.6, abs(row["Close"]-row["Open"]),
            color=color, zorder=2))

    x = range(len(df))
    # MA
    for w, c, lw in [(5,"#f59e0b",1.0),(20,"#8b5cf6",1.2),
                     (60,"#06b6d4",1.4),(120,"#10b981",1.4)]:
        col = f"MA{w}"
        if col in df.columns:
            ax1.plot(x, df[col], color=c, lw=lw, label=f"MA{w}", alpha=0.8)

    # 볼린저밴드
    if "BB_upper" in df.columns:
        ax1.plot(x, df["BB_upper"], color="#94a3b8", lw=0.8, ls="--", alpha=0.6)
        ax1.plot(x, df["BB_lower"], color="#94a3b8", lw=0.8, ls="--", alpha=0.6)
        ax1.fill_between(x, df["BB_upper"], df["BB_lower"],
                         alpha=0.05, color="#94a3b8")

    # 일목균형표 구름대
    if "span_a" in df.columns and "span_b" in df.columns:
        ax1.fill_between(x, df["span_a"], df["span_b"],
                         where=df["span_a"] >= df["span_b"],
                         alpha=0.15, color="#ef4444", label="구름대(양)")
        ax1.fill_between(x, df["span_a"], df["span_b"],
                         where=df["span_a"] < df["span_b"],
                         alpha=0.15, color="#3b82f6", label="구름대(음)")

    ax1.legend(loc="upper left", fontsize=8, ncol=4)
    ax1.set_xlim(-1, len(df))
    ax1.xaxis.set_visible(False)
    ax1.set_ylabel("가격", fontsize=9)
    ax1.grid(axis="y", alpha=0.3)

    # ── 2. RSI ────────────────────────────────
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    if "RSI" in df.columns:
        ax2.plot(x, df["RSI"],     color="#8b5cf6", lw=1.2, label="RSI")
        ax2.plot(x, df["RSI_sig"], color="#f59e0b", lw=0.9,
                 ls="--", label="Signal")
        ax2.axhline(70, color="#ef4444", lw=0.7, ls="--", alpha=0.6)
        ax2.axhline(30, color="#3b82f6", lw=0.7, ls="--", alpha=0.6)
        ax2.fill_between(x, df["RSI"], 70,
                         where=df["RSI"]>=70, alpha=0.15, color="#ef4444")
        ax2.fill_between(x, df["RSI"], 30,
                         where=df["RSI"]<=30, alpha=0.15, color="#3b82f6")
        ax2.set_ylim(0, 100)
        ax2.set_ylabel("RSI", fontsize=9)
        ax2.legend(loc="upper left", fontsize=8)
        ax2.grid(alpha=0.3)
    ax2.xaxis.set_visible(False)

    # ── 3. MACD ───────────────────────────────
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    if "MACDh" in df.columns:
        colors = ["#ef4444" if v >= 0 else "#3b82f6"
                  for v in df["MACDh"]]
        ax3.bar(x, df["MACDh"], color=colors, alpha=0.7, width=0.8)
        ax3.axhline(0, color="gray", lw=0.7)
        ax3.set_ylabel("MACD", fontsize=9)
        ax3.grid(axis="y", alpha=0.3)
    ax3.xaxis.set_visible(False)

    # ── 4. Stoch Slow ─────────────────────────
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    if "%K" in df.columns and "%D" in df.columns:
        ax4.plot(x, df["%K"], color="#8b5cf6", lw=1.2, label="%K")
        ax4.plot(x, df["%D"], color="#f59e0b", lw=0.9,
                 ls="--", label="%D")
        ax4.axhline(80, color="#ef4444", lw=0.7, ls="--", alpha=0.6)
        ax4.axhline(20, color="#3b82f6", lw=0.7, ls="--", alpha=0.6)
        ax4.set_ylim(0, 100)
        ax4.set_ylabel("Stoch", fontsize=9)
        ax4.legend(loc="upper left", fontsize=8)
        ax4.grid(alpha=0.3)

    # X축 날짜 레이블
    tick_step = max(1, len(df) // 10)
    ticks = list(range(0, len(df), tick_step))
    labels = [df.index[i].strftime("%m/%d") for i in ticks]
    ax4.set_xticks(ticks)
    ax4.set_xticklabels(labels, fontsize=8)

    plt.tight_layout()

    # Base64 변환
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode("utf-8")
    return img_b64

# ════════════════════════════════════════════
#  3. 섹터 분석
# ════════════════════════════════════════════
def analyze_sectors():
    print("\n  [섹터 분석] SPDR ETF 11개 로딩 중...")
    sector_data = {}
    for etf, label in SECTOR_ETFS.items():
        try:
            d_df, w_df = prepare_df(etf)
            if d_df is None:
                continue
            ws      = weekly_trend_score(w_df)
            rsi_val = round(d_df["RSI"].iloc[-1], 1) if pd.notna(d_df["RSI"].iloc[-1]) else 50
            cur     = d_df["Close"].iloc[-1]
            ret_1w  = round((cur / d_df["Close"].iloc[-5]  - 1) * 100, 2) if len(d_df) >= 5  else 0
            ret_1m  = round((cur / d_df["Close"].iloc[-21] - 1) * 100, 2) if len(d_df) >= 21 else 0
            ret_3m  = round((cur / d_df["Close"].iloc[-63] - 1) * 100, 2) if len(d_df) >= 63 else 0
            ret_6m  = round((cur / d_df["Close"].iloc[-126]- 1) * 100, 2) if len(d_df) >= 126 else 0

            # 기간별 강세/약세 판정 (True/False)
            bull_1w = ret_1w > 0
            bull_1m = ret_1m > 0
            bull_3m = ret_3m > 0
            bull_6m = ret_6m > 0

            # 강세 기간 카운트 (0~4)
            bull_count = sum([bull_1w, bull_1m, bull_3m, bull_6m])

            # 추세 유형 판정
            if bull_count == 4:
                trend_type = "완전상승"   # 전 기간 강세
            elif bull_count == 3:
                trend_type = "상승추세"
            elif bull_count == 2:
                if bull_1w and bull_1m:
                    trend_type = "단기반등"   # 장기는 약세, 단기만 강세
                elif bull_3m and bull_6m:
                    trend_type = "장기강세"   # 장기는 강세, 단기 조정
                else:
                    trend_type = "혼조"
            elif bull_count == 1:
                if bull_1w:
                    trend_type = "반등시도"   # 1주만 강세 → 일시 반등
                else:
                    trend_type = "약세전환"
            else:
                trend_type = "완전약세"

            # 섹터 가중치 (주봉 + 1달 기준)
            if ws >= 2 and ret_1m > 0 and rsi_val < 70:
                status, weight_add = "🟢 강세", 1
            elif ws >= 1 and ret_1m > -2:
                status, weight_add = "🟡 중립", 0
            elif ws <= -1 or ret_1m < -5:
                status, weight_add = "🔴 약세", -1
            else:
                status, weight_add = "⚪ 관망", 0

            sector_data[etf] = {
                "label": label, "score": ws, "rsi": rsi_val,
                "ret_1w": ret_1w, "ret_1m": ret_1m,
                "ret_3m": ret_3m, "ret_6m": ret_6m,
                "bull_1w": bull_1w, "bull_1m": bull_1m,
                "bull_3m": bull_3m, "bull_6m": bull_6m,
                "trend_type": trend_type,
                "status": status, "weight_add": weight_add,
            }
        except Exception as e:
            print(f"    [{etf}] 오류: {e}")
    sector_data = dict(sorted(sector_data.items(),
        key=lambda x: (x[1]["score"], x[1]["ret_1m"]), reverse=True))
    print(f"  섹터 분석 완료 ({len(sector_data)}개)")
    return sector_data

def get_sector_weight(ticker, sector_map, sector_data):
    """
    sector_map: {ticker: etf코드} — yfinance 자동 수집
    sector_data: {etf코드: {...}} — ETF 분석 결과
    """
    etf = get_etf_code(ticker, sector_map)
    if etf and etf in sector_data:
        return sector_data[etf]["weight_add"], sector_data[etf]["status"]
    return 0, "미분류"

def print_sector_summary(sector_data):
    print(f"\n  {'ETF':>5}  {'섹터':28}  {'주봉':>4}  {'RSI':>5}  "
          f"{'1주':>6}  {'1달':>6}  {'분기':>6}  {'반기':>6}  "
          f"{'추세유형':>8}  상태")
    print("  " + "─"*95)
    for etf, d in sector_data.items():
        # 기간별 색상 표시
        def fmt(val, bull):
            arrow = "▲" if bull else "▼"
            return f"{arrow}{abs(val):.1f}%"

        print(f"  {etf:>5}  {d['label']:28}  {d['score']:>+4}  "
              f"{d['rsi']:>5.1f}  "
              f"{fmt(d['ret_1w'], d['bull_1w']):>6}  "
              f"{fmt(d['ret_1m'], d['bull_1m']):>6}  "
              f"{fmt(d['ret_3m'], d['bull_3m']):>6}  "
              f"{fmt(d['ret_6m'], d['bull_6m']):>6}  "
              f"{d['trend_type']:>8}  {d['status']}")

# ════════════════════════════════════════════
#  4. 전략C — 전종목 스캔
# ════════════════════════════════════════════
def run_market_scan(sector_data=None, sector_map=None):
    print(f"\n{'='*60}")
    print(f"  시장 스캔 시작: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"  기준: 섹터별 최적 전략 자동 적용 (백테스팅 기반)")
    print(f"        XLK/XLC/XLY/XLV/XLP → 전략C | XLF → 전략D | 나머지 → 전략A")
    print(f"{'='*60}")

    # 티커 자동 수집
    sp500   = get_sp500_tickers_auto()
    ndaq100 = get_nasdaq100_tickers_auto()
    tickers = list(set(sp500 + ndaq100))
    print(f"  스캔 대상: {len(tickers)}종목")

    # SPY 수익률 미리 로드 (베타 계산용) ← 여기에 추가
    try:
        spy_df, _ = prepare_df("SPY")
        spy_returns = spy_df["Close"].pct_change().tail(60) if spy_df is not None else None
    except:
        spy_returns = None

    # 세 마녀의 날 정보 미리 로드 ← 추가
    tw_info  = get_triple_witching_info()
    vix_info = get_vix_info()  # ← 추가
    if tw_info["warning"]:
        print(f"  ⚠ {tw_info['label']}")
    if vix_info["warning"]:  # ← 추가
        print(f"  ⚠ {vix_info['label']}")

    # sector_map 없으면 자동 구축 (시간 소요 — 생략 가능)
    if sector_map is None:
        sector_map = {}

    results = []
    errors  = 0

    def _gauge(done, total, found, err):
        pct  = done / total * 100 if total else 0
        fill = min(pct, 100)
        clr  = "#10b981" if pct >= 70 else "#8b5cf6" if pct >= 30 else "#3b82f6"
        return HTML(f"""<div style="font-family:monospace;padding:8px 4px">
  <div style="background:#1e293b;border-radius:8px;overflow:hidden;height:22px;width:520px;border:1px solid #334155">
    <div style="background:linear-gradient(90deg,#3b82f6,{clr});width:{fill:.1f}%;height:100%"></div>
  </div>
  <span style="color:#94a3b8;font-size:12px;margin-top:4px;display:block">
    <b style="color:#f1f5f9;font-size:14px">{pct:.1f}%</b>
    ({done}/{total} 종목)
    | 🎯 신호 <b style="color:#10b981">{found}</b>종목 발견
    | ⚠ 오류 <b style="color:#ef4444">{err}</b>
  </span>
</div>""")
    _prog = display(_gauge(0, len(tickers), 0, 0), display_id=True)

    for i, ticker in enumerate(tickers):
        try:
            d_df, w_df = prepare_df(ticker)
            if d_df is None:
                continue
            ws = weekly_trend_score(w_df)
            sig_cnt,  sig_list  = count_buy_signals(d_df)
            sell_cnt, sell_list = count_sell_signals(d_df)

            sector_etf = get_etf_code(ticker, sector_map)
            strategy   = SECTOR_STRATEGY.get(sector_etf, DEFAULT_STRATEGY)
            params   = STRATEGY_PARAMS[strategy]

            if ws < params["weekly_min"]:
                continue
            if sig_cnt < 5:
                continue
            if params["need_cloud"]:
                r_chk = d_df.iloc[-1]
                if not (pd.notna(r_chk.get("span_a")) and
                        pd.notna(r_chk.get("span_b")) and
                        r_chk["Close"] > max(r_chk["span_a"], r_chk["span_b"])):
                    continue
            entry = round(d_df["Close"].iloc[-1], 2)
            # 52주 신고가 근접 여부
            high_52w = d_df["High"].tail(252).max()
            near_52w_high = entry >= high_52w * 0.80  # 신고가 20% 이내

            # 베타 계수 계산
            try:
                if spy_returns is not None and len(spy_returns) >= 60:
                    ret_stock = d_df["Close"].pct_change().tail(60)
                    _cov  = np.cov(ret_stock.values, spy_returns.values)[0][1]
                    _var  = np.var(spy_returns.values)
                    beta  = round(_cov / _var, 2) if _var > 0 else 1.0
                else:
                    beta = 1.0
            except:
                beta = 1.0
            etf_code   = get_etf_code(ticker, sector_map)
            industry   = get_industry(ticker, sector_map)
            trend_type = sector_data.get(etf_code, {}).get("trend_type", "미분류") if sector_data else "미분류"
            fund_cnt, fund_risks, fund_judge = check_fundamental(ticker, etf_code)
            # 성장주 스코어 추가
            growth_score, growth_grade, growth_details = check_growth_score(ticker, etf_code)


            stop, target, rr = calc_stop_target(d_df, entry)
            stop_pct   = round((stop   / entry - 1) * 100, 2)
            target_pct = round((target / entry - 1) * 100, 2)
            sec_add, sec_status = get_sector_weight(ticker, sector_map, sector_data or {})
            # 52주 신고가 근접 보정 (+1점)
            if near_52w_high:
                sec_add += 1
            # 성장주 스코어 보정 반영
            if growth_score >= 8:
                sec_add += 2
            elif growth_score >= 5:
                sec_add += 1

            adj_signals = sig_cnt + sec_add

            # IV는 보정점수 높은 종목만 계산
            if adj_signals >= 8:
                iv_info = get_option_iv(ticker)
            else:
                iv_info = {"iv": None, "pcr": None, "iv_label": "N/A",
                           "pcr_label": "N/A", "warning": False}

            if adj_signals < 8:
                continue

            results.append({
                "date":      datetime.now().strftime("%Y-%m-%d"),
                "ticker":    ticker,
                "price":     entry,
                "signals":   sig_cnt,
                "adj_sig":   adj_signals,
                "sell_cnt":  sell_cnt,
                "weekly":    ws,
                "sector":    sec_status,
                "trend_type": trend_type,
                "stop":      stop,
                "stop_pct":  stop_pct,
                "target":    target,
                "target_pct":target_pct,
                "rr":        rr,
                "buy_detail": " / ".join(sig_list),
                "sell_detail":" / ".join(sell_list) if sell_list else "없음",
                "strategy": strategy,
                "details":   " / ".join(sig_list),
                "fund_judge": fund_judge,
                "beta":         beta,
                "near_52w":     near_52w_high,
                "priority":     "⭐⭐" if adj_signals >= 10 else "⭐" if adj_signals >= 9 else "",
                "adx":          round(d_df["ADX"].iloc[-1], 1) if "ADX" in d_df.columns and pd.notna(d_df["ADX"].iloc[-1]) else 0,
                "fund_cnt":   fund_cnt,
                "fund_risks": " / ".join(fund_risks) if fund_risks else "없음",
                "growth_score": growth_score,
                "growth_grade": growth_grade,
                "growth_detail": " / ".join([f"{k}:{v}" for k,v in growth_details.items()]),
                "tw_warning":   tw_info["label"] if tw_info["warning"] else "",
                "tw_days":      tw_info["days_until"],
                "iv_label":  iv_info["iv_label"],
                "pcr_label": iv_info["pcr_label"],
                "iv_warning": iv_info["warning"],
                "vix_status": vix_info["status"],
                "vix_signal": vix_info["signal"],
                "industry":  industry,
            })
            _prog.update(_gauge(i + 1, len(tickers), len(results), errors))
        except Exception as e:
            errors += 1
            if errors <= 50:   # 오류 50개까지 출력
                print(f"  [{ticker}] 오류: {type(e).__name__}: {e}")

    _prog.update(_gauge(len(tickers), len(tickers), len(results), errors))
    results.sort(key=lambda x: (x["adj_sig"], x["weekly"]), reverse=True)
    print(f"\n  스캔 완료 → 신호 발생: {len(results)}종목 (오류: {errors})")
    return results

# ════════════════════════════════════════════
#  5. 포트폴리오 데일리 업데이트 (기존 동일)
# ════════════════════════════════════════════
def create_sample_portfolio():
    sample = pd.DataFrame([
        {"ticker":"CAT","buy_price":764.3021,"shares":10,"buy_date":"2026-01-01"},
        {"ticker":"CDNS","buy_price":332.28,"shares":60,"buy_date":"2026-01-01"},
        {"ticker":"GOOGL","buy_price":308.3147,"shares":4,"buy_date":"2026-01-01"},
        {"ticker":"JPM","buy_price":309.1727,"shares":3,"buy_date":"2026-01-01"},
        {"ticker":"NVDA","buy_price":186.6520,"shares":5,"buy_date":"2026-01-01"},
        {"ticker":"RKLB","buy_price":59.2025,"shares":42,"buy_date":"2026-01-01"},
        {"ticker":"TSLA","buy_price":416.1024,"shares":3,"buy_date":"2026-01-01"},
        {"ticker":"WMT","buy_price":129.1158,"shares":7,"buy_date":"2026-01-01"},


    ])
    sample.to_csv(PORTFOLIO_FILE, index=False, encoding="utf-8-sig")
    print(f"  샘플 포트폴리오 생성: {PORTFOLIO_FILE}")
    return sample

def _parse_krw(s):
    """'₩ 32,413,500' 또는 '32,413,500.00' → float, 실패 시 None"""
    if not s or str(s).strip() in ('', '#N/A', 'N/A', '-'):
        return None
    import re
    nums = re.sub(r'[₩\s,]', '', str(s))
    try:
        return float(nums)
    except Exception:
        return None

def _parse_num(s):
    """'23,259.70' → float, 실패 시 None"""
    if not s or str(s).strip() in ('', '#N/A', 'N/A', '-'):
        return None
    import re
    nums = re.sub(r'[,\s]', '', str(s))
    try:
        return float(nums)
    except Exception:
        return None

# 원자재 종목명 키워드
_RAW_MATERIAL_KEYWORDS = ['99.99', '원자재', '골드', 'gold', '실버', 'silver']

def _is_raw_material(name):
    nm = str(name).lower()
    return any(k in nm for k in _RAW_MATERIAL_KEYWORDS)


def load_unified_portfolio():
    """
    3개 시트에서 전체 자산 로드 및 분류
    반환: {
      "stocks":            [{"ticker","name","market","sector","shares","buy_price_krw","eval_krw","gain_krw","daily_krw"}],
      "raw_materials":     [{"name","shares","buy_price_krw","eval_krw"}],
      "deposits":          [{"type","bank","amount","rate","maturity","recv_after_tax","currency"}],
      "expired_deposits":  [{"type","bank","amount","recv_after_tax"}],
      "cash_accounts":     [{"label","bank","currency","amount_krw"}],
      "total_stock_krw":   float,
      "total_raw_krw":     float,
      "total_deposit_krw": float,
      "total_cash_krw":    float,
      "grand_total_krw":   float,
    }
    """
    import gspread
    from google.oauth2.service_account import Credentials
    import datetime

    today = datetime.date.today()
    result = {
        "stocks": [], "raw_materials": [],
        "deposits": [], "expired_deposits": [],
        "cash_accounts": [],
        "total_stock_krw": 0.0, "total_raw_krw": 0.0,
        "total_deposit_krw": 0.0, "total_cash_krw": 0.0,
        "grand_total_krw": 0.0,
    }

    try:
        scopes = [
            "https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive",
        ]
        if os.path.exists(GSHEET_KEY_FILE):
            creds = Credentials.from_service_account_file(GSHEET_KEY_FILE, scopes=scopes)
        else:
            key_dict = json.loads(os.environ.get("GSHEET_JSON", "{}"))
            creds = Credentials.from_service_account_info(key_dict, scopes=scopes)
        gc = gspread.authorize(creds)
        sh = gc.open_by_key(GSHEET_ID)

        # ── [1] 주식현황상세 ──────────────────────────────
        # row1: 분류 헤더(병합), row2: 컬럼명, row3+: 데이터
        # 컬럼 인덱스 (0-based):
        #  0:시장 1:종목명 2:섹터 3:코드 4:보유량 5:매수평단가
        #  8:매수총액(원) 9:평균환율
        #  14:평가총액값 15:평가총액(원) 17:총수익값 21:일간수익값
        try:
            ws1   = sh.worksheet(GSHEET_SHEET)
            rows1 = ws1.get_all_values()
            exchange_rate = _parse_num(rows1[0][9]) or 1471.0  # row1 col J = 환율

            for r in rows1[2:]:  # row3부터 데이터
                if not r or not any(r):
                    continue
                # 컬럼 안전 접근
                def _c(idx, default=''):
                    return r[idx] if idx < len(r) else default

                market    = _c(0)
                name      = _c(1)
                sector    = _c(2)
                ticker    = _c(3)
                shares    = _parse_num(_c(4)) or 0
                buy_price = _parse_num(_c(5))   # 국내=원, 미국=달러
                buy_total_krw = _parse_num(_c(8))

                # 평가총액(원): col15, 없으면 col14 활용, 없으면 매수단가 기반 추정
                eval_krw = _parse_krw(_c(15))
                if eval_krw is None:
                    eval_val = _parse_num(_c(14))
                    if eval_val is not None:
                        # col13 통화 확인: '₩' → KRW, '$' → USD×환율
                        if '₩' in _c(13) or market == '국내':
                            eval_krw = eval_val
                        else:
                            eval_krw = eval_val * exchange_rate
                if eval_krw is None and buy_price and shares:
                    # fallback: 매수단가 × 보유량
                    if market == '국내':
                        eval_krw = buy_price * shares
                    else:
                        eval_krw = buy_price * shares * exchange_rate

                buy_price_krw = buy_total_krw or (
                    (buy_price * shares) if buy_price and market == '국내'
                    else (buy_price * shares * exchange_rate) if buy_price
                    else None
                )

                gain_krw  = _parse_krw(_c(17)) if len(r) > 17 else None
                daily_krw = _parse_krw(_c(21)) if len(r) > 21 else None

                row_data = {
                    "ticker":        ticker,
                    "name":          name,
                    "market":        market,
                    "sector":        sector,
                    "shares":        shares,
                    "buy_price_krw": buy_price_krw,
                    "eval_krw":      eval_krw or 0.0,
                    "gain_krw":      gain_krw,
                    "daily_krw":     daily_krw,
                }

                if _is_raw_material(name):
                    result["raw_materials"].append(row_data)
                    result["total_raw_krw"] += row_data["eval_krw"]
                else:
                    result["stocks"].append(row_data)
                    result["total_stock_krw"] += row_data["eval_krw"]

            print(f"  [통합자산] 주식 {len(result['stocks'])}종목 / 원자재 {len(result['raw_materials'])}종목 로드")
        except Exception as e:
            print(f"  ⚠ 주식현황상세 로드 실패: {e}")

        # ── [2] 예적금내역 ──────────────────────────────
        # 컬럼: 0:구분 1:은행 2:가입일 3:이율 4:기간 5:금액
        #        7:통화 8:만기수령액세전 9:만기수령액세후
        #       11:만기일 12:환율 13:만기수령액통화
        try:
            ws2   = sh.worksheet(GSHEET_SHEET_DEPOSIT)
            rows2 = ws2.get_all_values()
            for r in rows2[1:]:  # row2부터 데이터
                if not r or not any(r) or not r[0]:
                    continue
                def _c2(idx, default=''):
                    return r[idx] if idx < len(r) else default

                dep_type = _c2(0)
                bank     = _c2(1)
                if not dep_type and not bank: continue  # 빈 행 skip
                amount   = _parse_num(_c2(5)) or 0
                currency = _c2(7) or 'KRW'
                # recv_세후(col9)는 '₩9,109,204' 형식 → _parse_krw 사용
                # recv_통화(col13)는 '9,109,203.82' 형식 → _parse_num 사용 (더 정확)
                recv_cur = _parse_num(_c2(13)) or 0
                recv_tax = recv_cur if recv_cur else (_parse_krw(_c2(9)) or 0)
                maturity_str = _c2(11)
                exrate   = _parse_num(_c2(12)) or 1.0

                # 만기일 파싱
                maturity_date = None
                try:
                    import datetime as _dt
                    maturity_date = _dt.date.fromisoformat(maturity_str)
                except Exception:
                    pass

                # 만기수령액(원) 계산
                recv_krw = recv_tax * exrate  # exrate=1이면 동일

                row_dep = {
                    "type":          dep_type,
                    "bank":          bank,
                    "amount":        amount,
                    "rate":          _c2(3),
                    "maturity":      maturity_str,
                    "maturity_date": maturity_date,
                    "recv_after_tax": recv_krw,
                    "currency":      currency,
                }

                if maturity_date and maturity_date <= today:
                    # 만기 완료 → 현금으로 분류
                    result["expired_deposits"].append(row_dep)
                    result["total_cash_krw"] += recv_krw
                else:
                    result["deposits"].append(row_dep)
                    result["total_deposit_krw"] += recv_krw

            print(f"  [통합자산] 예적금 {len(result['deposits'])}건 / 만기완료 {len(result['expired_deposits'])}건 로드")
        except Exception as e:
            print(f"  ⚠ 예적금내역 로드 실패: {e}")

        # ── [3] 입출금통장 ──────────────────────────────
        # row1: 계좌구분, row2: 은행명, row3: 통화, row4+: 날짜별 잔액
        # 최신 행(마지막 행) 사용
        try:
            ws3   = sh.worksheet(GSHEET_SHEET_CASH)
            rows3 = ws3.get_all_values()
            if len(rows3) >= 4:
                header1 = rows3[0]   # 계좌명
                header2 = rows3[1]   # 은행명
                header3 = rows3[2]   # 통화
                latest  = rows3[-1]  # 최신 데이터 행

                for col_idx in range(2, len(header1)):
                    label    = header1[col_idx] if col_idx < len(header1) else ""
                    bank     = header2[col_idx] if col_idx < len(header2) else ""
                    currency = header3[col_idx] if col_idx < len(header3) else "KRW"
                    amount   = _parse_num(latest[col_idx]) if col_idx < len(latest) else None

                    if not label and not bank:
                        continue

                    result["cash_accounts"].append({
                        "label":      label,
                        "bank":       bank,
                        "currency":   currency,
                        "amount_krw": amount or 0.0,
                    })
                    result["total_cash_krw"] += amount or 0.0

            print(f"  [통합자산] 입출금통장 {len(result['cash_accounts'])}계좌 로드")
        except Exception as e:
            print(f"  ⚠ 입출금통장 로드 실패: {e}")

    except Exception as e:
        print(f"  ⚠ 통합자산 로드 전체 실패: {e}")

    result["grand_total_krw"] = (
        result["total_stock_krw"] + result["total_raw_krw"]
        + result["total_deposit_krw"] + result["total_cash_krw"]
    )
    print(f"  [통합자산] 총 자산 {result['grand_total_krw']:,.0f}원")
    return result


def format_asset_overview_html(unified):
    """통합 자산 현황 HTML 섹션 생성 (v2)"""
    if not unified:
        return ""
    import datetime as _dt

    def _fmt_krw(v):
        if v is None: return "—"
        v = float(v)
        if abs(v) >= 1e8:  return f"{v/1e8:.2f}억원"
        if abs(v) >= 1e4:  return f"{v/1e4:.0f}만원"
        return f"₩{v:,.0f}"

    def _fmt_krw_full(v):
        if v is None: return "—"
        return f"₩{float(v):,.0f}"

    def _pct(part, total):
        if not total: return "0.0%"
        return f"{float(part)/float(total)*100:.1f}%"

    grand = unified.get("grand_total_krw", 0) or 1
    stock = unified.get("total_stock_krw", 0)
    raw   = unified.get("total_raw_krw", 0)
    dep   = unified.get("total_deposit_krw", 0)
    cash  = unified.get("total_cash_krw", 0)
    cash_dep_total = cash + dep  # 현금+예적금 합계

    # ── 요약 카드 ──
    def _card(label, val, bg, bc):
        return (
            f'<div style="flex:1;min-width:150px;background:{bg};border-radius:10px;'
            f'padding:14px 18px;border-left:4px solid {bc};box-sizing:border-box">'
            f'<div style="font-size:11px;color:#6b7280;margin-bottom:4px">{label}</div>'
            f'<div style="font-size:17px;font-weight:700;color:#1e293b">{_fmt_krw(val)}</div>'
            f'<div style="font-size:11px;color:{bc};margin-top:2px">{_pct(val, grand)}</div>'
            f'</div>'
        )
    cards_html = (
        _card("💵 현금·예적금", cash_dep_total, "#f0fdf4", "#16a34a") +
        _card("🥇 원자재",      raw,            "#fffbeb", "#d97706") +
        _card("📈 주식",        stock,          "#faf5ff", "#7c3aed") +
        f'<div style="flex:1;min-width:150px;background:#f1f5f9;border-radius:10px;'
        f'padding:14px 18px;border-left:4px solid #1e40af;box-sizing:border-box">'
        f'<div style="font-size:11px;color:#6b7280;margin-bottom:4px">🏦 총 자산</div>'
        f'<div style="font-size:17px;font-weight:700;color:#1e40af">{_fmt_krw(grand)}</div>'
        f'<div style="font-size:11px;color:#6b7280;margin-top:2px">100%</div></div>'
    )
    summary_cards = (
        f'<div style="display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px">'
        f'{cards_html}</div>'
    )

    # ══ 현금·예적금 통합 테이블 ══
    # 입출금 계좌
    cash_rows = ""
    for acc in unified.get("cash_accounts", []):
        amt = acc["amount_krw"]
        cash_rows += (
            f'<tr style="border-bottom:1px solid #f1f5f9">'
            f'<td style="padding:8px 12px;color:#374151">{acc["label"]}</td>'
            f'<td style="padding:8px 12px;color:#6b7280">{acc["bank"]}</td>'
            f'<td style="padding:8px 12px;color:#6b7280">—</td>'
            f'<td style="padding:8px 12px;color:#6b7280">—</td>'
            f'<td style="padding:8px 12px;text-align:right;font-weight:600">'
            f'{_fmt_krw_full(amt)}</td>'
            f'</tr>'
        )
    # ACTIVE 예적금 (만기 남은 것만)
    today = _dt.date.today()
    for d in unified.get("deposits", []):
        md = d.get("maturity_date")
        if md and md <= today:
            continue  # 이미 만기 → skip
        type_badge = (
            '<span style="background:#dbeafe;color:#1d4ed8;padding:1px 6px;'
            'border-radius:4px;font-size:10px">예금</span>'
            if d["type"] == "예금" else
            '<span style="background:#dcfce7;color:#15803d;padding:1px 6px;'
            'border-radius:4px;font-size:10px">적금</span>'
        )
        cash_rows += (
            f'<tr style="border-bottom:1px solid #f1f5f9;background:#fafffe">'
            f'<td style="padding:8px 12px">{type_badge}</td>'
            f'<td style="padding:8px 12px;color:#6b7280">{d["bank"]}</td>'
            f'<td style="padding:8px 12px;color:#6b7280;text-align:center">{d["rate"]}</td>'
            f'<td style="padding:8px 12px;color:#6b7280;text-align:center">{d["maturity"]}</td>'
            f'<td style="padding:8px 12px;text-align:right;font-weight:600">'
            f'{_fmt_krw_full(d["recv_after_tax"])}</td>'
            f'</tr>'
        )

    cash_section = (
        f'<h4 style="color:#16a34a;margin:16px 0 6px;font-size:14px">'
        f'💵 현금·예적금'
        f'<span style="font-size:12px;color:#6b7280;font-weight:400;margin-left:8px">'
        f'합계 {_fmt_krw(cash_dep_total)}</span></h4>'
        f'<table style="width:100%;border-collapse:collapse;font-size:13px">'
        f'<thead><tr style="background:#f0fdf4">'
        f'<th style="padding:8px 12px;text-align:left">구분</th>'
        f'<th style="padding:8px 12px;text-align:left">기관</th>'
        f'<th style="padding:8px 12px;text-align:center">이율</th>'
        f'<th style="padding:8px 12px;text-align:center">만기일</th>'
        f'<th style="padding:8px 12px;text-align:right">잔액 / 만기수령액</th>'
        f'</tr></thead>'
        f'<tbody>{cash_rows}</tbody>'
        f'</table>'
    ) if cash_rows else ""

    # ══ 원자재 테이블 (평가금액·손익 제거) ══
    raw_rows = ""
    for rm in unified.get("raw_materials", []):
        raw_rows += (
            f'<tr style="border-bottom:1px solid #f1f5f9">'
            f'<td style="padding:8px 12px;font-weight:600">{rm["name"]}</td>'
            f'<td style="padding:8px 12px;text-align:right">{rm["shares"]:,.0f}개</td>'
            f'<td style="padding:8px 12px;text-align:right">{_fmt_krw_full(rm["buy_price_krw"])}</td>'
            f'</tr>'
        )
    raw_section = (
        f'<h4 style="color:#d97706;margin:16px 0 6px;font-size:14px">'
        f'🥇 원자재'
        f'<span style="font-size:12px;color:#6b7280;font-weight:400;margin-left:8px">'
        f'합계 {_fmt_krw(raw)}</span></h4>'
        f'<table style="width:100%;border-collapse:collapse;font-size:13px">'
        f'<thead><tr style="background:#fffbeb">'
        f'<th style="padding:8px 12px;text-align:left">종목명</th>'
        f'<th style="padding:8px 12px;text-align:right">보유량</th>'
        f'<th style="padding:8px 12px;text-align:right">매수총액</th>'
        f'</tr></thead>'
        f'<tbody>{raw_rows}</tbody>'
        f'</table>'
    ) if raw_rows else ""

    # ══ 주식 테이블 (코드·일간손익 제거 + ETF 통합) ══
    # ETF 통합 키 매핑
    ETF_GROUP = {
        'S&P500':   ['s&p500', 's&p 500'],
        '나스닥100': ['나스닥100', '나스닥'],
        '다우존스':  ['다우존스'],
    }
    def _etf_group_key(name):
        nm = name.lower()
        for key, keywords in ETF_GROUP.items():
            if any(k in nm for k in keywords):
                return key
        return None

    # 국내 ETF 통합
    grouped = {}   # key → {name, market, shares, buy_price_krw, eval_krw, gain_krw}
    solo_stocks = []
    for s in unified.get("stocks", []):
        if s["market"] == "국내":
            gk = _etf_group_key(s["name"])
            if gk:
                if gk not in grouped:
                    grouped[gk] = {
                        "name": gk + " ETF (국내 합산)",
                        "market": "국내ETF",
                        "shares": 0,
                        "buy_price_krw": 0.0,
                        "eval_krw": 0.0,
                        "gain_krw": 0.0,
                    }
                g = grouped[gk]
                g["shares"]        += s.get("shares", 0)
                g["buy_price_krw"] += s.get("buy_price_krw") or 0
                g["eval_krw"]      += s.get("eval_krw") or 0
                if s.get("gain_krw") is not None:
                    g["gain_krw"] = (g["gain_krw"] or 0) + s["gain_krw"]
                continue
        solo_stocks.append(s)

    # 정렬: 국내ETF 통합 → 국내 개별 → 미국
    display_stocks = (
        sorted(grouped.values(), key=lambda x: -x["eval_krw"]) +
        sorted([s for s in solo_stocks if s["market"] == "국내"], key=lambda x: -x["eval_krw"]) +
        sorted([s for s in solo_stocks if s["market"] != "국내"], key=lambda x: -x["eval_krw"])
    )

    stock_rows = ""
    for s in display_stocks:
        gain = s.get("gain_krw")
        gain_html = ""
        if gain is not None and gain != 0:
            color = "#16a34a" if gain >= 0 else "#dc2626"
            sign  = "+" if gain >= 0 else ""
            gain_html = f'<span style="color:{color}">{sign}{_fmt_krw_full(gain)}</span>'

        market = s.get("market", "")
        if market == "국내ETF":
            badge = ('<span style="background:#e0f2fe;color:#0369a1;padding:1px 6px;'
                     'border-radius:4px;font-size:10px">국내ETF</span>')
        elif market == "국내":
            badge = ('<span style="background:#dbeafe;color:#1d4ed8;padding:1px 6px;'
                     'border-radius:4px;font-size:10px">국내</span>')
        else:
            badge = ('<span style="background:#fce7f3;color:#be185d;padding:1px 6px;'
                     'border-radius:4px;font-size:10px">미국</span>')

        stock_rows += (
            f'<tr style="border-bottom:1px solid #f1f5f9">'
            f'<td style="padding:8px 12px">{badge}</td>'
            f'<td style="padding:8px 12px;font-weight:600">{s["name"]}</td>'
            f'<td style="padding:8px 12px;text-align:right">{s["shares"]:,.0f}</td>'
            f'<td style="padding:8px 12px;text-align:right;font-weight:600">'
            f'{_fmt_krw_full(s["eval_krw"])}</td>'
            f'<td style="padding:8px 12px;text-align:right">{gain_html}</td>'
            f'</tr>'
        )

    stock_section = (
        f'<h4 style="color:#7c3aed;margin:16px 0 6px;font-size:14px">'
        f'📈 주식'
        f'<span style="font-size:12px;color:#6b7280;font-weight:400;margin-left:8px">'
        f'합계 {_fmt_krw(stock)}</span></h4>'
        f'<table style="width:100%;border-collapse:collapse;font-size:13px">'
        f'<thead><tr style="background:#faf5ff">'
        f'<th style="padding:8px 12px;text-align:left">시장</th>'
        f'<th style="padding:8px 12px;text-align:left">종목명</th>'
        f'<th style="padding:8px 12px;text-align:right">보유량</th>'
        f'<th style="padding:8px 12px;text-align:right">평가금액</th>'
        f'<th style="padding:8px 12px;text-align:right">손익</th>'
        f'</tr></thead>'
        f'<tbody>{stock_rows}</tbody>'
        f'</table>'
    ) if stock_rows else ""

    _today_str = today.strftime("%Y-%m-%d")
    return (
        f'<h3 style="color:#1e293b;border-bottom:2px solid #1e293b;'
        f'padding-bottom:6px;margin-top:20px">'
        f'🏦 전체 자산 현황'
        f'<small style="color:#6b7280;font-size:12px;font-weight:400;margin-left:8px">'
        f'({_today_str} 기준 | 단위: 원)</small></h3>'
        f'{summary_cards}'
        f'{cash_section}'
        f'{raw_section}'
        f'{stock_section}'
        f'<p style="font-size:11px;color:#9ca3af;margin-top:8px">'
        f'※ 만기완료 예적금은 표시 제외 | 국내 ETF는 지수별로 통합 표시 | '
        f'미국주식 환율 반영</p>'
    )


def run_portfolio_update(sector_map=None, sector_data=None):
    print(f"\n{'='*60}")
    print(f"  포트폴리오 업데이트: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(f"{'='*60}")
     # Google Sheets에서 포트폴리오 로드
    try:
        import gspread
        from google.oauth2.service_account import Credentials
        scopes = [
            "https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive",
        ]
        import json
        if os.path.exists(GSHEET_KEY_FILE):
            # 로컬 환경
            creds = Credentials.from_service_account_file(
                        GSHEET_KEY_FILE, scopes=scopes)
        else:
            # Streamlit Cloud 환경
            key_dict = json.loads(os.environ.get("GSHEET_JSON", "{}"))
            creds = Credentials.from_service_account_info(
                        key_dict, scopes=scopes)
        gc        = gspread.authorize(creds)
        sh        = gc.open_by_key(GSHEET_ID)
        ws        = sh.worksheet(GSHEET_SHEET)
        # 주식현황상세: row1=분류헤더, row2=컬럼명, row3+=데이터
        rows_raw = ws.get_all_values()
        exchange_rate = _parse_num(rows_raw[0][9]) or 1471.0
        records = []
        for r in rows_raw[2:]:
            if not r or not any(r) or not r[3]: continue
            def _c(idx, d=""):
                return r[idx] if idx < len(r) else d
            eval_krw = _parse_krw(_c(15))
            if eval_krw is None:
                ev = _parse_num(_c(14))
                if ev:
                    eval_krw = ev if (_c(0)=="국내" or "₩" in _c(13)) else ev * exchange_rate
            shares_v = _parse_num(_c(4)) or 1
            bp_raw   = _parse_num(_c(5))
            if eval_krw is None and bp_raw:
                eval_krw = bp_raw * shares_v * (1 if _c(0)=="국내" else exchange_rate)
            buy_price = (eval_krw / shares_v) if (eval_krw and shares_v) else (bp_raw or 0)
            # 원자재는 포트폴리오 기술분석 제외
            if _is_raw_material(_c(1)): continue
            records.append({
                "ticker":    _c(3),
                "name":      _c(1),
                "buy_price": round(buy_price, 2) if buy_price else 0,
                "shares":    shares_v,
                "buy_date":  "",
                "market":    _c(0),
            })
        portfolio = pd.DataFrame(records)
        print(f"  Google Sheets 로드 완료: {len(portfolio)}종목 (주식현황상세)")
    except Exception as e:
        print(f"  ⚠ Google Sheets 로드 실패: {e}")
        print(f"  → CSV 폴백으로 전환")
        if not os.path.exists(PORTFOLIO_FILE):
            portfolio = create_sample_portfolio()
        else:
            portfolio = pd.read_csv(PORTFOLIO_FILE, encoding="utf-8-sig")
    required = {"ticker","buy_price","shares"}
    if not required.issubset(portfolio.columns):
        return []
    port_results = []
    for _, row in portfolio.iterrows():
        ticker    = row["ticker"]
        buy_price = float(row["buy_price"])
        shares    = float(row["shares"])
        buy_date  = str(row["buy_date"])
        try:
            d_df, w_df = prepare_df(ticker)
            if d_df is None:
                continue
            cur_price   = round(d_df["Close"].iloc[-1], 2)
            pnl_pct     = round((cur_price / buy_price - 1) * 100, 2)
            eval_amt    = round(cur_price * shares, 2)
            pnl_amt     = round(eval_amt - buy_price * shares, 2)
            sig_cnt, sig_list = count_buy_signals(d_df)
            ws          = weekly_trend_score(w_df)
            stop, target, rr = calc_stop_target(d_df, cur_price)
            etf_code_port  = get_etf_code(ticker, sector_map)
            fund_cnt, fund_risks, fund_judge = check_fundamental(ticker, etf_code_port)
            stop_dist   = round((stop   / cur_price - 1) * 100, 2)
            target_dist = round((target / cur_price - 1) * 100, 2)
            try:
                hold_days = (datetime.now() -
                             datetime.strptime(buy_date, "%Y-%m-%d")).days
            except:
                hold_days = 0
            _, sec_status = get_sector_weight(
                ticker, sector_map or {}, sector_data or {})
            # 목표가 대비 현재 수익 진행률
            target_progress = (cur_price - buy_price) / (target - buy_price) * 100 \
                              if target > buy_price else 0

            # 섹터 추세 확인
            _, sec_status = get_sector_weight(ticker, sector_map or {}, sector_data or {})
            sec_bull = "강세" in sec_status

            # 펀더멘털 판정
            fund_ok = "✅" in fund_judge

            # 세 마녀의 날 정보
            tw_info  = get_triple_witching_info()
            vix_info = get_vix_info()  # ← 추가
            earn_info = get_earnings_info(ticker)

            # 어닝 발표일 정보
            earn_info = get_earnings_info(ticker)

            # 옵션 IV 정보
            iv_info = get_option_iv(ticker)

            try:
                memo_val = row["memo"] if "memo" in row.index else ""
                memo = str(memo_val).strip() if pd.notna(memo_val) and str(memo_val) != "nan" else ""
            except:
                memo = ""

            # 어닝 발표 임박 경고 (최우선) ← 여기에 추가
            if earn_info["days_until_earnings"] is not None and \
               earn_info["days_until_earnings"] <= 3:
                status = f"🚨 어닝 {earn_info['days_until_earnings']}일 전 — 변동성 주의"

            elif tw_info["is_today"]:
                status = "🚨 세 마녀의 날 — 변동성 극대화"

            elif tw_info["warning"]:
                status = f"⚠ 세 마녀의 날 {tw_info['days_until']}일 전"

            elif "재진입대기" in memo:
                if sig_cnt >= 3 and ws >= 2:
                    status = "🟣 재진입 신호 발생 — 매수 검토"
                elif sig_cnt >= 1:
                    status = "🔵 재진입 대기 (신호 약함)"
                else:
                    status = "🔵 재진입 대기 (신호 없음)"

            elif cur_price <= stop:
                status = "🔴 손절 고려"

            elif target_progress >= 80:
                if fund_ok and ws >= 2 and sec_bull:
                    status = "🟡 1/3 익절 후 장기보유"
                else:
                    status = "🟡 1/3 익절 고려"

            elif target_progress >= 50:
                if fund_ok and ws >= 2 and sec_bull:
                    status = "🟡 분할익절 고려 (추세 양호)"
                else:
                    status = "🟠 분할익절 고려 (추세 점검)"

            elif cur_price <= stop * 1.03:
                status = "🟠 손절가 근접 (모니터링)"

            elif sig_cnt >= 3 and ws >= 2 and pnl_pct < 5:
                status = "🟣 추가매수 고려"

            elif fund_ok and ws >= 2 and sec_bull and sig_cnt >= 2:
                status = "🟢 장기보유 유지"

            elif not sec_bull and ws <= 0:
                status = "🔵 추세 약화 (모니터링)"

            elif sig_cnt == 0 and pnl_pct < 0:
                status = "🟠 신호 소멸 (모니터링)"

            else:
                status = "⚪ 보유 유지"

            port_results.append({
                "ticker": ticker, "buy_price": buy_price,
                "cur_price": cur_price, "pnl_pct": pnl_pct,
                "pnl_amt": pnl_amt, "eval_amt": eval_amt,
                "hold_days": hold_days, "signals": sig_cnt,
                "weekly": ws, "sector": sec_status,
                "stop": stop, "target": target,
                "stop_dist": stop_dist, "target_dist": target_dist,
                "status": status,
                "memo": memo,
                "target_progress": round(target_progress, 1),
                "sig_details": " / ".join(sig_list) if sig_list else "없음",
                "fund_judge": fund_judge,
                "fund_cnt":   fund_cnt,
                "fund_risks": " / ".join(fund_risks) if fund_risks else "없음",
                "next_earnings":    earn_info["next_earnings"],
                "surprise_label":   earn_info["surprise_label"],
                "earn_warning":     earn_info["warning"],
                "tw_label":         tw_info["label"] if tw_info["warning"] else "",
                "iv_label":  iv_info["iv_label"],
                "pcr_label": iv_info["pcr_label"],
                "vix_status": vix_info["status"],
                "vix_signal": vix_info["signal"],
                "industry":  get_industry(ticker, sector_map),
            })
        except Exception as e:
            print(f"  [{ticker}] 오류: {e}")
    if port_results:
        total_eval = sum(r["eval_amt"] for r in port_results)
        total_cost = sum(
            r["buy_price"] * portfolio[
                portfolio["ticker"]==r["ticker"]]["shares"].values[0]
            for r in port_results)
        total_pnl = total_eval - total_cost
        total_pct = (total_eval / total_cost - 1) * 100 if total_cost > 0 else 0
        print(f"\n  총 평가금액: ${total_eval:,.2f}  |  "
              f"총 손익: ${total_pnl:+,.2f} ({total_pct:+.2f}%)")
    return port_results

# ════════════════════════════════════════════
#  6. 이메일 발송
# ════════════════════════════════════════════
def send_email(scan_results, port_results, sector_data=None, vc_results=None, earn_results=None, sector_map=None, unified_portfolio=None):
    sector_map = sector_map or {}
    # ── 통합 자산 현황 HTML
    asset_overview_html = format_asset_overview_html(unified_portfolio) if unified_portfolio else ""
    today = datetime.now().strftime("%Y-%m-%d")

    # ── 분기 실적 헬퍼 함수 (earnings table) ─────────
    def _fmt_fin(v):
        if v is None:
            return '<span style="color:#9ca3af">—</span>'
        try: v = float(v)
        except: return str(v)
        if abs(v) >= 1000: return f'${v/1000:.2f}B'
        elif abs(v) >= 1:  return f'${v:.0f}M'
        else:              return f'${v*1000:.0f}K'

    def _fmt_pct_cell(v, bold=False):
        if v is None:
            return '<span style="color:#9ca3af">—</span>'
        try: v = float(v)
        except: return str(v)
        color = '#16a34a' if v > 0 else '#dc2626' if v < 0 else '#6b7280'
        fw    = 'font-weight:700;' if bold else ''
        sign  = '+' if v > 0 else ''
        return f'<span style="color:{color};{fw}">{sign}{v:.1f}%</span>'

    SECTOR_EXCEPTIONS_LABEL = {
        'XLK': '고PER 허용 (성장주 섹터)', 'XLC': '고PER 허용 (성장주 섹터)',
        'XLY': '고PER 허용 (소비재 성장)', 'XLV': 'R&D 비용 제외 기준',
        'XLF': '은행·보험 PBR 기준',       'XLI': '해당없음',
        'XLP': '해당없음',                  'XLE': '에너지 EV/EBITDA 기준',
        'XLB': '해당없음',                  'XLU': '배당수익률 보조지표',
        'XLRE': 'FFO 기준 (리츠)',          '':    '해당없음',
    }

    # 섹터별 평균 밸류에이션 레퍼런스 ← 여기에 추가
    SECTOR_AVG_VAL = {
        "XLK":  {"PER": 32, "ForwardPER": 28, "PSR": 7,  "PEG": 2.1},
        "XLC":  {"PER": 25, "ForwardPER": 22, "PSR": 4,  "PEG": 1.8},
        "XLY":  {"PER": 24, "ForwardPER": 20, "PSR": 1.5,"PEG": 1.9},
        "XLP":  {"PER": 22, "ForwardPER": 19, "PSR": 1.2,"PEG": 3.2},
        "XLF":  {"PER": 13, "ForwardPER": 11, "PSR": 3,  "PEG": 1.4},
        "XLV":  {"PER": 24, "ForwardPER": 20, "PSR": 4,  "PEG": 2.3},
        "XLI":  {"PER": 22, "ForwardPER": 18, "PSR": 1.5,"PEG": 2.0},
        "XLB":  {"PER": 17, "ForwardPER": 15, "PSR": 1.2,"PEG": 1.8},
        "XLE":  {"PER": 12, "ForwardPER": 10, "PSR": 0.8,"PEG": 1.2},
        "XLU":  {"PER": 18, "ForwardPER": 16, "PSR": 2.5,"PEG": 2.8},
        "XLRE": {"PER": 28, "ForwardPER": 24, "PSR": 4,  "PEG": 2.5},
    }

    # 세 마녀의 날 경고 배너
    tw_info = get_triple_witching_info()
    tw_banner = ""
    if tw_info["warning"]:
        tw_banner = f"""
<div style="background:#fef2f2;border:2px solid #dc2626;border-radius:8px;
            padding:12px 16px;margin-bottom:16px;font-weight:700;color:#dc2626">
  {tw_info["label"]} ({tw_info["date"]}) — 변동성 급증 구간, 신규 진입 주의!
</div>"""

    # VIX 경고 배너 ← tw_info 블록 밖에 위치
    vix_info = get_vix_info()
    vix_banner = ""
    if vix_info["vix"] is not None:
        vix_banner = f"""
<div style="background:{vix_info['color']}15;border:2px solid {vix_info['color']};
            border-radius:8px;padding:12px 16px;margin-bottom:16px;
            font-weight:700;color:{vix_info['color']}">
  📊 시장 심리 — {vix_info['label']} | MA20: {vix_info['vix_ma20']} | 
  추세: {vix_info['vix_trend']} | {vix_info['signal']}
</div>"""

    # ── 섹터별 현황 ──────────────────────────
    sector_html = ""
    if sector_data:
        rows = ""
        for etf, d in sector_data.items():
            if   "강세" in d["status"]: bg="#f0fdf4"; sc="#16a34a"
            elif "약세" in d["status"]: bg="#fef2f2"; sc="#dc2626"
            elif "중립" in d["status"]: bg="#fffbeb"; sc="#d97706"
            else:                       bg="#f9fafb"; sc="#6b7280"
            sc2 = "#16a34a" if d["score"]>=2 else "#dc2626" if d["score"]<=-1 else "#d97706"
            r1c = "#16a34a" if d["ret_1m"]>0 else "#dc2626"
            r3c = "#16a34a" if d["ret_3m"]>0 else "#dc2626"
            r1w = "#16a34a" if d["ret_1w"]>0 else "#dc2626"
            r6m = "#16a34a" if d["ret_6m"]>0 else "#dc2626"
            tc  = {"완전상승":"#16a34a","상승추세":"#16a34a","장기강세":"#d97706",
                   "단기반등":"#d97706","혼조":"#d97706","반등시도":"#dc2626",
                   "약세전환":"#dc2626","완전약세":"#dc2626"}.get(d["trend_type"],"#6b7280")
            rows += f"""<tr style="background:{bg}">
  <td style="padding:10px 16px;font-weight:600;color:#1e40af">{etf}</td>
  <td style="padding:10px 16px">{d["label"]}</td>
  <td style="padding:10px 16px;text-align:center;color:{sc2};font-weight:600">{d["score"]:+d}</td>
  <td style="padding:10px 16px;text-align:center">{d["rsi"]:.1f}</td>
  <td style="padding:10px 16pxx;text-align:center;color:{r1w};font-weight:600">{d["ret_1w"]:+.1f}%</td>
  <td style="padding:10px 16px;text-align:center;color:{r1c};font-weight:600">{d["ret_1m"]:+.1f}%</td>
  <td style="padding:10px 16pxx;text-align:center;color:{r3c};font-weight:600">{d["ret_3m"]:+.1f}%</td>
  <td style="padding:10px 16pxx;text-align:center;color:{r6m};font-weight:600">{d["ret_6m"]:+.1f}%</td>
  <td style="padding:10px 16pxx;text-align:center;font-size:12px;font-weight:600;color:{tc}">{d["trend_type"]}</td>
  <td style="padding:10px 16px;text-align:center;font-weight:700;color:#4338ca">
    {SECTOR_STRATEGY.get(etf, DEFAULT_STRATEGY)}</td>
  <td style="padding:10px 16pxx;text-align:center;color:{sc};font-weight:600">{d["status"]}</td>
</tr>"""
        sector_html = f"""
<h3 style="color:#1e40af;border-bottom:2px solid #1e40af;padding-bottom:6px;margin-top:20px">
  📊 산업 섹터별 현황
</h3>
<table style="width:100%;border-collapse:collapse;font-size:14px;margin-bottom:20px">
<thead><tr style="background:#1e40af;color:white">
  <th style="padding:11px 16px;text-align:left">ETF</th>
  <th style="padding:11px 16px;text-align:left">섹터</th>
  <th style="padding:11px 16px;text-align:center">주봉점수</th>
  <th style="padding:11px 16px;text-align:center">RSI</th>
  <th style="padding:11px 16px;text-align:center">1주</th>
  <th style="padding:11px 16px;text-align:center">1달</th>
  <th style="padding:11px 16px;text-align:center">분기</th>
  <th style="padding:11px 16px;text-align:center">반기</th>
  <th style="padding:11px 16px;text-align:center">추세유형</th>
  <th style="padding:11px 16px;text-align:center">적용전략</th>
  <th style="padding:11px 16px;text-align:center">상태</th>
</tr></thead><tbody>{rows}</tbody></table>
<p style="font-size:12px;color:#6b7280;margin-top:6px;line-height:1.8">
  ※ 적용 전략 기준 (백테스팅 결과 기반)<br>
  ▪ <b>전략A</b> — 신호 3개 이상 (주봉 무관) | 안정적 종목 적합 | 에너지·소재·산업재·유틸리티·부동산<br>
  ▪ <b>전략C</b> — 주봉 점수 +2 이상 + 신호 3개 이상 | 고변동 종목 노이즈 필터링 | 기술·커뮤니케이션·임의소비재·헬스케어·필수소비재<br>
  ▪ <b>전략D</b> — 주봉 점수 +2 이상 + 구름대 위 + 신호 3개 이상 | 가장 보수적 | 금융
</p>"""

    # ── 상세 산업별 현황 ─────────────────────
    from collections import defaultdict
    ind_scan_map = defaultdict(list)
    for r in scan_results:
        ind = r.get("industry", "") or "미분류"
        ind_scan_map[ind].append(r)
    ind_port_map = defaultdict(list)
    for r in port_results:
        ind = r.get("industry", "") or "미분류"
        ind_port_map[ind].append(r)
    all_inds_html = sorted(set(list(ind_scan_map.keys()) + list(ind_port_map.keys())))

    # industry → ETF 코드 매핑
    def _ind_to_etf(ind):
        for r in (scan_results + port_results):
            if r.get("industry","") == ind:
                _ts = {r2["ticker"].upper(): get_etf_code(r2["ticker"], sector_map)
                   for r2 in (scan_results + port_results)}
                return _ts.get(r["ticker"].upper(), "")
        return ""

    def _fmt_ret(val):
        if val is None: return "—"
        c = "#16a34a" if val > 0 else "#dc2626"
        arrow = "▲" if val > 0 else "▼"
        return f'<span style="color:{c};font-weight:600">{arrow}{abs(val):.1f}%</span>'

    industry_rows_html = ""
    for ind in all_inds_html:
        sc = ind_scan_map.get(ind, [])
        pc = ind_port_map.get(ind, [])
        scan_tickers = " ".join(f'<span style="background:#dcfce7;color:#166534;padding:2px 6px;border-radius:4px;font-size:11px;margin:1px;display:inline-block">{r["ticker"]}</span>' for r in sc)
        port_tickers = " ".join(f'<span style="background:#ede9fe;color:#6d28d9;padding:2px 6px;border-radius:4px;font-size:11px;margin:1px;display:inline-block">{r["ticker"]}</span>' for r in pc)
        scan_cell = scan_tickers if sc else '<span style="color:#9ca3af">—</span>'
        port_cell = port_tickers if pc else '<span style="color:#9ca3af">—</span>'
        row_bg = "#f0fdf4" if sc else "#faf5ff" if pc else "#f9fafb"
        etf_k  = _ind_to_etf(ind)
        sd     = (sector_data or {}).get(etf_k, {})
        rsi_v  = f"{sd['rsi']:.1f}" if sd.get("rsi") is not None else "—"
        rsi_c  = "#dc2626" if sd.get("rsi",50)>70 else "#3b82f6" if sd.get("rsi",50)<30 else "#374151"
        r1w_v  = _fmt_ret(sd.get("ret_1w"))
        r1m_v  = _fmt_ret(sd.get("ret_1m"))
        r3m_v  = _fmt_ret(sd.get("ret_3m"))
        r6m_v  = _fmt_ret(sd.get("ret_6m"))
        industry_rows_html += f"""<tr style="background:{row_bg}">
  <td style="padding:12px 15px;font-weight:600;color:#1e293b">{ind}</td>
  <td style="padding:12px 15px;text-align:center;font-weight:700;color:{rsi_c}">{rsi_v}</td>
  <td style="padding:12px 15px;text-align:center">{r1w_v}</td>
  <td style="padding:12px 15px;text-align:center">{r1m_v}</td>
  <td style="padding:12px 15px;text-align:center">{r3m_v}</td>
  <td style="padding:12px 15px;text-align:center">{r6m_v}</td>
  <td style="padding:12px 15px">{scan_cell}</td>
  <td style="padding:12px 15px">{port_cell}</td>
</tr>"""

    industry_html = ""
    if all_inds_html:
        industry_html = f"""
<h3 style="color:#0f766e;border-bottom:2px solid #0f766e;padding-bottom:6px;margin-top:20px">
  📂 상세 산업별 현황
</h3>
<table style="width:100%;border-collapse:collapse;font-size:13px;margin-bottom:16px">
<thead><tr style="background:#0f766e;color:white">
  <th style="padding:14px 15px;text-align:left">세부 산업</th>
  <th style="padding:14px 15px;text-align:center">RSI</th>
  <th style="padding:14px 15px;text-align:center">1주</th>
  <th style="padding:14px 15px;text-align:center">1달</th>
  <th style="padding:14px 15px;text-align:center">분기</th>
  <th style="padding:14px 15px;text-align:center">반기</th>
  <th style="padding:14px 15px;text-align:left">신호 종목</th>
  <th style="padding:14px 15px;text-align:left">포트폴리오</th>
</tr></thead><tbody>{industry_rows_html}</tbody></table>
<p style="font-size:12px;color:#6b7280;margin-top:4px">
  ※ RSI/수익률은 해당 산업의 상위 ETF 섹터 기준 | 🟢 신호 발생 | 🟣 보유 중
</p>"""

        # ── 매수 시그널 상세 테이블 ───────────────
    if scan_results:
        scan_rows = ""
        for i, r in enumerate(scan_results):
            bg       = "#f0fdf4" if i%2==0 else "white"
            rr_c     = "#16a34a" if r["rr"]>=2 else "#dc2626"
            sec      = r.get("sector","미분류")
            sec_c    = "#16a34a" if "강세" in sec else "#dc2626" if "약세" in sec                        else "#d97706" if "중립" in sec else "#9ca3af"
            sp       = r.get("stop_pct",  round((r["stop"]  /r["price"]-1)*100,1))
            tp       = r.get("target_pct",round((r["target"]/r["price"]-1)*100,1))
            sell_cnt = r.get("sell_cnt", 0)
            sell_det = r.get("sell_detail","없음")
            buy_det  = r.get("buy_detail", r.get("details",""))
            growth_g = r.get("growth_grade", "")
            growth_c = "#dc2626" if "🔥" in growth_g else "#d97706" if "⭐" in growth_g else "#6b7280"
            date_str = r.get("date", today)
            scan_rows += f"""<tr style="background:{bg}">
  <td style="padding:13px 15px;text-align:center;font-size:14px">{r.get("priority","")}</td>
  <td style="padding:13px 15px;font-weight:700;color:#1e40af">{r["ticker"]}</td>
  <td style="padding:13px 15px;text-align:right">${r["price"]:.2f}</td>
  <td style="padding:13px 15px;text-align:center">
    <span style="background:#dcfce7;color:#16a34a;font-weight:700;padding:1px 7px;border-radius:10px">{r["signals"]}개</span></td>
  <td style="padding:13px 15px;text-align:center">
    <span style="background:#fee2e2;color:#dc2626;font-weight:700;padding:1px 7px;border-radius:10px">{sell_cnt}개</span></td>
  <td style="padding:13px 15px;text-align:center">
    <span style="background:#e0e7ff;color:#4338ca;font-weight:700;padding:1px 7px;border-radius:10px">{r["adj_sig"]}개</span></td>
  <td style="padding:13px 15px;text-align:center;font-weight:600;
      color:{"#16a34a" if r["weekly"]>=2 else "#d97706"}">{r["weekly"]:+d}</td>
  <td style="padding:13px 15px;text-align:center;font-weight:700;color:#4338ca">[{r.get("strategy","C")}]</td>
  <td style="padding:13px 15px;text-align:center;font-size:12px;color:{sec_c};font-weight:600">{sec}</td>
  <td style="padding:13px 15px;text-align:center;color:#dc2626;font-weight:600">
    ${r["stop"]:.2f}<br><small style="color:#dc2626">{sp:+.1f}%</small></td>
  <td style="padding:13px 15px;text-align:center;color:#16a34a;font-weight:600">
    ${r["target"]:.2f}<br><small style="color:#16a34a">{tp:+.1f}%</small></td>
  <td style="padding:13px 15px;text-align:center;font-weight:700;color:{growth_c}">{growth_g}</td>
  <td style="padding:13px 15px;font-size:12px;color:#15803d;
    word-break:break-word;overflow-wrap:break-word">{buy_det}</td>
  <td style="padding:13px 15px;font-size:12px;color:#dc2626;
    word-break:break-word;overflow-wrap:break-word">{sell_det}</td>
</tr>"""
        scan_section = f"""
<h3 style="color:#16a34a;border-bottom:2px solid #16a34a;padding-bottom:6px">
  🔍 매수 시그널 종목 — {len(scan_results)}종목
  <small style="color:#6b7280;font-size:13px;font-weight:400">
    (S&P500+나스닥100 | 주봉 +2이상 + 신호 3개이상)
  </small>
</h3>
<p style="font-size:12px;color:#6b7280;margin-bottom:8px;line-height:1.8">
  ※ 보정점수(adj) 계산 기준<br>
  ▪ <b>기본 매수신호</b> — 구름대 위 / MA정배열 / RSI상향 / MACD양전환 / Stoch GC / BB하단 / 거래량급증 / RSI기울기 / OBV상승 / ADX강세 (각 +1점)<br>
  ▪ <b>섹터 가중치</b> — 🟢 강세 +1점 | 🟡 중립 0점 | 🔴 약세 -1점<br>
  ▪ <b>52주 신고가</b> — 현재가가 52주 신고가의 80% 이상 → +1점<br>
  ▪ <b>성장주 스코어</b> — 시총/PEG/PSR/PER/EPS성장/OPM/목표주가/내부자매수 종합 → 8점↑ +2점 | 5점↑ +1점
</p>
<table style="width:100%;border-collapse:collapse;font-size:13px;table-layout:fixed">
<thead><tr style="background:#16a34a;color:white">
  <th style="padding:14px 15px;text-align:center">순위</th>
  <th style="padding:14px 15px;text-align:left">종목</th>
  <th style="padding:14px 15px;text-align:right">현재가</th>
  <th style="padding:14px 15px;text-align:center">매수신호</th>
  <th style="padding:14px 15px;text-align:center">매도신호</th>
  <th style="padding:14px 15px;text-align:center">보정</th>
  <th style="padding:14px 15px;text-align:center">주봉</th>
  <th style="padding:14px 15px;text-align:center">전략</th>
  <th style="padding:14px 15px;text-align:center">섹터</th>
  <th style="padding:14px 15px;text-align:center">손절가(%)</th>
  <th style="padding:14px 15px;text-align:center">목표가(%)</th>
  <th style="padding:14px 15px;text-align:center">성장성</th>
  <th style="padding:14px 15px;text-align:left;width:220px">매수 근거</th>
  <th style="padding:14px 15px;text-align:left;width:175px">매도 근거</th>
</tr></thead><tbody>{scan_rows}</tbody></table>
<p style="font-size:12px;color:#6b7280;margin-top:6px;line-height:1.8">
  ※ 성장성 스코어 계산 기준 (0~15점)<br>
  ▪ <b>시가총액</b> — $10B 미만 소형주 +3점 | $100B 미만 중형주 +1점<br>
  ▪ <b>PEG</b> — 1.0 미만 +3점 (성장대비 저평가) | 2.0 미만 +1점 (적정)<br>
  ▪ <b>PSR</b> — 섹터 저평가 기준 미만 +2점 | 섹터 적정 기준 미만 +1점<br>
  ▪ <b>PER</b> — 섹터 저평가 기준 미만 +2점 | 섹터 적정 기준 미만 +1점<br>
  ▪ <b>Forward PER</b> — 섹터 저평가 기준 미만 +2점 | Trailing 대비 개선 +1점<br>
  ▪ <b>EPS 성장률</b> — 30%↑ +3점 | 10%↑ +1점<br>
  ▪ <b>OPM (영업이익률)</b> — 섹터 기준 이상 + 3%p↑ +2점 | 개선중 +1점<br>
  ▪ <b>목표주가</b> — 현재가 대비 20%↑ +2점 | 10%↑ +1점 (애널리스트 3인↑)<br>
  ▪ <b>내부자 매수</b> — 최근 순매수 +1점<br>
  ▪ 🔥 최우선 (8점↑) | ⭐ 유망 (5점↑) | ✅ 관심 (3점↑) | ➖ 보통
</p>"""
    else:
        scan_section = """
<h3 style="color:#16a34a;border-bottom:2px solid #16a34a;padding-bottom:6px">
  🔍 매수 시그널 종목
</h3>
<p style="color:#6b7280">오늘은 신호 발생 종목이 없습니다.</p>"""

    # ── 밸류에이션 상세 테이블 ────────────────
    val_rows = ""
    for i, r in enumerate(scan_results):
        bg       = "#f8fff8" if i%2==0 else "white"
        ticker   = r["ticker"]
        etf_code = r.get("sector_etf") or TICKER_SECTOR.get(ticker.upper(), "")
        sec_avg  = SECTOR_AVG_VAL.get(etf_code, {})
        gd       = r.get("growth_detail", "")

        import re
        def get_val(detail_str, key):
            for item in detail_str.split(" / "):
                if key+":" in item:
                    val = item.replace(key+":", "").strip()
                    nums = re.findall(r"[-\d.]+", val)
                    return nums[0] if nums else "N/A"
            return "N/A"

        per_val     = get_val(gd, "PER")
        fwd_per_val = get_val(gd, "ForwardPER")
        psr_val     = get_val(gd, "PSR")
        peg_val     = get_val(gd, "PEG")
        eps_val     = get_val(gd, "EPS성장")
        opm_val     = get_val(gd, "OPM")

        def val_color(val_str, avg):
            try:
                v = float(val_str)
                if avg and v < avg * 0.8: return "#16a34a"
                if avg and v > avg * 1.2: return "#dc2626"
                return "#d97706"
            except:
                return "#9ca3af"

        per_c     = val_color(per_val,     sec_avg.get("PER"))
        fwd_per_c = val_color(fwd_per_val, sec_avg.get("ForwardPER"))
        psr_c     = val_color(psr_val,     sec_avg.get("PSR"))
        peg_c     = val_color(peg_val,     sec_avg.get("PEG"))
        grade     = r.get("growth_grade", "")
        grade_c   = "#dc2626" if "🔥" in grade else "#d97706" if "⭐" in grade else "#6b7280"

        val_rows += f"""<tr style="background:{bg}">
  <td style="padding:10px 12px;font-weight:700;color:#1e40af">{ticker}</td>
  <td style="padding:10px 12px;text-align:center;font-weight:700;color:{grade_c}">{grade}</td>
  <td style="padding:10px 12px;text-align:center;color:{per_c};font-weight:600">{per_val}<br><small style="color:#9ca3af">섹터평균:{sec_avg.get("PER","N/A")}</small></td>
  <td style="padding:10px 12px;text-align:center;color:{fwd_per_c};font-weight:600">{fwd_per_val}<br><small style="color:#9ca3af">섹터평균:{sec_avg.get("ForwardPER","N/A")}</small></td>
  <td style="padding:10px 12px;text-align:center;color:{psr_c};font-weight:600">{psr_val}<br><small style="color:#9ca3af">섹터평균:{sec_avg.get("PSR","N/A")}</small></td>
  <td style="padding:10px 12px;text-align:center;color:{peg_c};font-weight:600">{peg_val}<br><small style="color:#9ca3af">섹터평균:{sec_avg.get("PEG","N/A")}</small></td>
  <td style="padding:10px 12px;text-align:center">{eps_val}</td>
  <td style="padding:10px 12px;text-align:center">{opm_val}</td>
</tr>"""

    val_section = f"""
<h3 style="color:#0891b2;border-bottom:2px solid #0891b2;
           padding-bottom:6px;margin-top:24px">
  📊 밸류에이션 상세 — 매수 시그널 종목
  <small style="color:#6b7280;font-size:13px;font-weight:400">
    (섹터 평균 대비 🟢 저평가 | 🟡 적정 | 🔴 고평가)
  </small>
</h3>
<table style="width:100%;border-collapse:collapse;font-size:13px">
<thead><tr style="background:#0891b2;color:white">
  <th style="padding:11px 12px;text-align:left">종목</th>
  <th style="padding:11px 12px;text-align:center">성장성</th>
  <th style="padding:11px 12px;text-align:center">PER</th>
  <th style="padding:11px 12px;text-align:center">Forward PER</th>
  <th style="padding:11px 12px;text-align:center">PSR</th>
  <th style="padding:11px 12px;text-align:center">PEG</th>
  <th style="padding:11px 12px;text-align:center">EPS성장률</th>
  <th style="padding:11px 12px;text-align:center">OPM</th>
</tr></thead><tbody>{val_rows}</tbody></table>
<p style="font-size:12px;color:#6b7280;margin-top:6px;line-height:1.8">
  ※ 섹터 평균 대비 20%↑ 저평가 🟢 | 적정 🟡 | 20%↑ 고평가 🔴
</p>"""

    # ── 펀더멘털 체크리스트 — 매수 시그널 ──────
    fund_scan_rows = ""
    for i, r in enumerate(scan_results):
        bg       = "#f8faff" if i%2==0 else "white"
        judge    = r.get("fund_judge", "조회실패")
        cnt      = r.get("fund_cnt", 0)
        risks    = r.get("fund_risks", "없음")
        etf_code = r.get("sector_etf") or TICKER_SECTOR.get(r["ticker"].upper(), "")
        exc_items= SECTOR_EXCEPTIONS_LABEL.get(etf_code, "해당없음")
        jc       = "#16a34a" if "✅" in judge else \
                   "#d97706" if "⚠"  in judge else \
                   "#dc2626" if "🔴" in judge else "#9ca3af"
        fund_scan_rows += f"""<tr style="background:{bg}">
  <td style="padding:10px 16pxx;font-weight:700;color:#1e40af">{r["ticker"]}</td>
  <td style="padding:10px 16px;text-align:center;font-weight:700;color:{jc}">{judge}({cnt}/8)</td>
  <td style="padding:10px 16px;font-size:12px;color:#dc2626">{risks}</td>
  <td style="padding:10px 16px;font-size:12px;color:#9ca3af">{exc_items}</td>
</tr>"""
    fund_scan_footnote = """<p style="font-size:12px;color:#6b7280;margin-top:6px;line-height:1.8">
  ※ 펀더멘털 체크리스트 항목 설명<br>
  ▪ <b>현금&lt;총부채</b> — 보유 현금이 총부채보다 적음 → 유동성 위기 가능성<br>
  ▪ <b>매출채권↑&gt;매출↑</b> — 매출채권 증가율이 매출보다 5%p↑ → 외상 매출, 현금 유입 불확실<br>
  ▪ <b>재고↑이익↓</b> — 재고 15%↑ 또는 순이익 마이너스 → 수요 감소 신호<br>
  ▪ <b>단기부채&gt;현금</b> — 단기 상환 의무 > 보유 현금 → 채무 상환 능력 부족<br>
  ▪ <b>무형자산N%</b> — 무형자산이 총자산 50% 초과 → 자산가치 변동성 증가<br>
  ▪ <b>기타자산N%</b> — 기타장기자산이 총자산 50% 초과 → 자산 구성 불투명<br>
  ▪ <b>우선주존재</b> — 우선주 발행 → 일반 주주보다 우선주 투자자에게 유리한 구조<br>
  ▪ <b>자본잠식</b> — 자기자본 음수 → 누적 손실로 자본 잠식, 상장폐지 위험<br>
  ▪ ✅ 양호(0개) &nbsp;|&nbsp; ⚠ 주의(1~2개) &nbsp;|&nbsp; 🔴 위험(3개 이상)
</p>"""
    fund_scan_section = f"""
<h3 style="color:#7c3aed;border-bottom:2px solid #7c3aed;
           padding-bottom:6px;margin-top:24px">
  🔬 펀더멘털 체크리스트 — 매수 시그널 종목
  <small style="color:#6b7280;font-size:13px;font-weight:400">
    (대차대조표 기준 8가지 위험 항목 | 섹터 특성 예외 반영)
  </small>
</h3>
<table style="width:100%;border-collapse:collapse;font-size:13px">
<thead><tr style="background:#7c3aed;color:white">
  <th style="padding:11px 16px;text-align:left">종목</th>
  <th style="padding:11px 16px;text-align:center">판정</th>
  <th style="padding:11px 16px;text-align:left">위험 항목</th>
  <th style="padding:11px 16px;text-align:left">섹터 예외 적용</th>
</tr></thead><tbody>{fund_scan_rows}</tbody></table>"""

    # ── 포트폴리오 현황 ───────────────────────
    if port_results:
        total_eval = sum(r["eval_amt"] for r in port_results)
        total_pnl  = sum(r["pnl_amt"]  for r in port_results)
        total_cost = total_eval - total_pnl
        total_pct  = (total_pnl/total_cost*100) if total_cost>0 else 0
        port_rows  = ""
        for i, r in enumerate(port_results):
            bg    = "#faf5ff" if i%2==0 else "white"
            pc    = "#16a34a" if r["pnl_pct"]>=0 else "#dc2626"
            port_rows += f"""<tr style="background:{bg}">
  <td style="padding:13px 15px;font-weight:700;color:#7c3aed">{r["ticker"]}</td>
  <td style="padding:13px 15px;text-align:right">${r["buy_price"]:.2f}</td>
  <td style="padding:13px 15px;text-align:right">${r["cur_price"]:.2f}</td>
  <td style="padding:13px 15px;text-align:center;font-weight:700;color:{pc}">{r["pnl_pct"]:+.2f}%</td>
  <td style="padding:13px 15px;text-align:right;color:{pc}">${r["pnl_amt"]:+,.2f}</td>
  <td style="padding:13px 15px;text-align:center;font-size:12px">{r.get("sector","미분류")}</td>
  <td style="padding:13px 15px;text-align:center;font-size:11px;color:#0f766e">{r.get("industry","—") or "—"}</td>
  <td style="padding:10px 12px;text-align:center;color:#dc2626;font-weight:600">
    ${r["stop"]:.2f}<br><small>{r["stop_dist"]:+.1f}%</small></td>
  <td style="padding:10px 12px;text-align:center;color:#16a34a;font-weight:600">
    ${r["target"]:.2f}<br><small>{r["target_dist"]:+.1f}%</small></td>
  <td style="padding:10px 12px;text-align:center;font-weight:600;
      color:{"#16a34a" if r.get("target_progress",0)>=50 else "#6b7280"}">
    {r.get("target_progress",0):.1f}%</td>
  <td style="padding:13px 15px;text-align:center">{r["signals"]}개</td>
  <td style="padding:10px 12px;text-align:center;font-size:12px;color:#d97706">
    {r.get("next_earnings","N/A")}<br>
    <small>{r.get("surprise_label","")}</small></td>
  <td style="padding:10px 12px;text-align:center;font-size:12px;color:#dc2626;font-weight:600">
    {r.get("earn_warning","")}{r.get("tw_label","")}</td>
  <td style="padding:10px 12px;text-align:center;font-size:12px;color:#7c3aed">
    {r.get("iv_label","N/A")}<br>
    <small>{r.get("pcr_label","")}</small></td>
  <td style="padding:13px 15px;font-size:12px;color:#6b7280">{r.get("memo","")}</td>
  <td style="padding:13px 15px">{r["status"]}</td>
</tr>"""
        port_section = f"""
<h3 style="color:#7c3aed;border-bottom:2px solid #7c3aed;padding-bottom:6px;margin-top:24px">
  💼 포트폴리오 현황 — {len(port_results)}종목
</h3>
<p style="font-size:14px;margin-bottom:8px">
  총 평가: <b>${total_eval:,.2f}</b> | 손익:
  <b style="color:{"#16a34a" if total_pnl>=0 else "#dc2626"}">
    ${total_pnl:+,.2f} ({total_pct:+.2f}%)</b>
</p>
<table style="width:100%;border-collapse:collapse;font-size:13px">
<thead><tr style="background:#7c3aed;color:white">
  <th style="padding:14px 15px;text-align:left">종목</th>
  <th style="padding:14px 15px;text-align:right">매수가</th>
  <th style="padding:14px 15px;text-align:right">현재가</th>
  <th style="padding:14px 15px;text-align:center">수익률</th>
  <th style="padding:14px 15px;text-align:right">평가손익</th>
  <th style="padding:14px 15px;text-align:center">섹터</th>
  <th style="padding:14px 15px;text-align:center">세부산업</th>
  <th style="padding:14px 15px;text-align:center">손절가(%)</th>
  <th style="padding:14px 15px;text-align:center">목표가(%)</th>
  <th style="padding:14px 15px;text-align:center">목표진행률</th>
  <th style="padding:14px 15px;text-align:center">신호</th>
  <th style="padding:14px 15px;text-align:center">어닝</th>
  <th style="padding:14px 15px;text-align:center">이벤트</th>
  <th style="padding:14px 15px;text-align:center">옵션</th>
  <th style="padding:14px 15px;text-align:left">메모</th>
  <th style="padding:14px 15px;text-align:left">상태</th>
</tr></thead><tbody>{port_rows}</tbody></table>"""
    else:
        port_section = """
<h3 style="color:#7c3aed;border-bottom:2px solid #7c3aed;padding-bottom:6px;margin-top:24px">
  💼 포트폴리오 현황
</h3>
<p style="color:#6b7280">포트폴리오 데이터가 없습니다.</p>"""

# ── 펀더멘털 체크리스트 — 포트폴리오 ────────
    fund_port_rows = ""
    for i, r in enumerate(port_results):
        bg       = "#f0f9ff" if i%2==0 else "white"
        judge    = r.get("fund_judge", "조회실패")
        cnt      = r.get("fund_cnt", 0)
        risks    = r.get("fund_risks", "없음")
        etf_code = r.get("sector_etf") or TICKER_SECTOR.get(r["ticker"].upper(), "")
        exc_items= SECTOR_EXCEPTIONS_LABEL.get(etf_code, "해당없음")
        jc       = "#16a34a" if "✅" in judge else \
                   "#d97706" if "⚠"  in judge else \
                   "#dc2626" if "🔴" in judge else "#9ca3af"
        fund_port_rows += f"""<tr style="background:{bg}">
  <td style="padding:10px 16px;font-weight:700;color:#7c3aed">{r["ticker"]}</td>
  <td style="padding:10px 16px;text-align:center;font-weight:700;color:{jc}">{judge}({cnt}/8)</td>
  <td style="padding:10px 16px;font-size:12px;color:#dc2626">{risks}</td>
  <td style="padding:10px 16px;font-size:12px;color:#9ca3af">{exc_items}</td>
</tr>"""
    fund_port_footnote = """<p style="font-size:12px;color:#6b7280;margin-top:6px;line-height:1.8">
  ※ 펀더멘털 체크리스트 항목 설명<br>
  ▪ <b>현금&lt;총부채</b> — 보유 현금이 총부채보다 적음 → 유동성 위기 가능성<br>
  ▪ <b>매출채권↑&gt;매출↑</b> — 매출채권 증가율이 매출보다 5%p↑ → 외상 매출, 현금 유입 불확실<br>
  ▪ <b>재고↑이익↓</b> — 재고 15%↑ 또는 순이익 마이너스 → 수요 감소 신호<br>
  ▪ <b>단기부채&gt;현금</b> — 단기 상환 의무 > 보유 현금 → 채무 상환 능력 부족<br>
  ▪ <b>무형자산N%</b> — 무형자산이 총자산 50% 초과 → 자산가치 변동성 증가<br>
  ▪ <b>기타자산N%</b> — 기타장기자산이 총자산 50% 초과 → 자산 구성 불투명<br>
  ▪ <b>우선주존재</b> — 우선주 발행 → 일반 주주보다 우선주 투자자에게 유리한 구조<br>
  ▪ <b>자본잠식</b> — 자기자본 음수 → 누적 손실로 자본 잠식, 상장폐지 위험<br>
  ▪ ✅ 양호(0개) &nbsp;|&nbsp; ⚠ 주의(1~2개) &nbsp;|&nbsp; 🔴 위험(3개 이상)
</p>"""

    # ── 보정 6개 이상 종목 차트 생성 ─────────
    chart_section = ""
    chart_tickers = [r for r in scan_results if r.get("adj_sig", 0) >= 7][:3]
    if chart_tickers:
        chart_html = ""
        for r in chart_tickers:
            try:
                d_df, _ = prepare_df(r["ticker"])
                if d_df is not None:
                    img_b64 = generate_chart_base64(r["ticker"], d_df)
                    if img_b64:
                        chart_html += f"""
<div style="margin-bottom:20px">
  <p style="font-size:13px;font-weight:600;color:#1e40af;margin-bottom:6px">
    {r["ticker"]} — 현재가 ${r["price"]:.2f} | 
    매수신호 {r["signals"]}개 | 보정 {r["adj_sig"]}개 | 
    {r.get("strategy","C")} 전략 | {r.get("sector","미분류")}
  </p>
  <img src="data:image/png;base64,{img_b64}" 
       style="width:100%;max-width:900px;border:1px solid #e5e7eb;
              border-radius:8px">
</div>"""
            except Exception as e:
                chart_html += f'<p style="color:#9ca3af">[{r["ticker"]}] 차트 생성 실패: {e}</p>'

        chart_section = f"""
<h3 style="color:#dc2626;border-bottom:2px solid #dc2626;
           padding-bottom:6px;margin-top:24px">
  📊 주요 매수 후보 차트 — 보정신호 6개 이상 종목
  <small style="color:#6b7280;font-size:12px;font-weight:400">
    (일봉 180일 | MA/볼린저밴드/일목구름/RSI/MACD/Stoch)
  </small>
</h3>
{chart_html}"""

    fund_port_section = f"""
<h3 style="color:#0891b2;border-bottom:2px solid #0891b2;
           padding-bottom:6px;margin-top:24px">
  🔬 펀더멘털 체크리스트 — 내 포트폴리오
  <small style="color:#6b7280;font-size:13px;font-weight:400">
    (대차대조표 기준 8가지 위험 항목 | 섹터 특성 예외 반영)
  </small>
</h3>
<table style="width:100%;border-collapse:collapse;font-size:13px">
<thead><tr style="background:#0891b2;color:white">
  <th style="padding:11px 16px;text-align:left">종목</th>
  <th style="padding:11px 16px;text-align:center">판정</th>
  <th style="padding:11px 16px;text-align:left">위험 항목</th>
  <th style="padding:11px 16px;text-align:left">섹터 예외 적용</th>
</tr></thead><tbody>{fund_port_rows}</tbody></table>"""



    # ── 포트폴리오 분기별 실적 HTML ──────────────
    earnings_html = ""
    if earn_results:
        earn_sections = ""
        for tk, d in earn_results.items():
            qs = d["quarters"]
            if not qs: continue
            # 헤더 행
            q_headers = "".join(
                f'<th style="padding:9px 12px;text-align:right;white-space:nowrap'
                f';background:#1e40af;color:white">{q["label"]}</th>' for q in qs
            )
            def _earn_row(label, vals, bg="#f9fafb", bold=False):
                fw = "font-weight:700;" if bold else ""
                _NA_SPAN = '<span style="color:#9ca3af">—</span>'
                cells = "".join(
                    f'<td style="padding:8px 12px;text-align:right;{fw}">'
                    f'{_fmt_fin(v) if v is not None else _NA_SPAN}'
                    f'</td>' for v in vals
                )
                return (f'<tr style="background:{bg}">'
                        f'<td style="padding:8px 12px;{fw}white-space:nowrap">{label}</td>'
                        f'{cells}</tr>')

            def _earn_pct_row(label, vals, bg="#f9fafb", bold=False):
                cells = "".join(
                    f'<td style="padding:8px 12px;text-align:right">{_fmt_pct_cell(v, bold)}</td>'
                    for v in vals
                )
                fw = "font-weight:700;" if bold else ""
                return (f'<tr style="background:{bg}">'
                        f'<td style="padding:8px 12px;{fw}white-space:nowrap">{label}</td>'
                        f'{cells}</tr>')

            body_rows = ""
            body_rows += _earn_row("📊 매출 합계",    [q["rev"]    for q in qs], bg="#dbeafe", bold=True)
            if d["has_geo"]:
                body_rows += _earn_row("↳ 국내(US)",   [q["dom"]   for q in qs], bg="#f0fdf4")
                body_rows += _earn_row("↳ 해외(Intl)", [q["intl"]  for q in qs], bg="#f0fdf4")
            body_rows += _earn_row("💰 영업이익",      [q["op_inc"] for q in qs], bg="#dcfce7", bold=True)
            body_rows += _earn_pct_row("영업이익률",   [q["op_margin"] for q in qs])

            earn_sections += f"""
<h4 style="color:#1e40af;margin:16px 0 6px">{d.get("company", d.get("name", tk))} ({tk})
  <span style="font-size:11px;color:#6b7280;font-weight:400">단위: {d.get("currency", "USD(M)")}</span>
</h4>
<div style="overflow-x:auto">
<table style="width:100%;border-collapse:collapse;font-size:12px;min-width:600px">
<thead><tr style="background:#1e40af;color:white">
  <th style="padding:9px 12px;text-align:left">항목</th>{q_headers}
</tr></thead>
<tbody>{body_rows}</tbody>
</table></div>"""

        earnings_html = f"""
<h3 style="color:#1e40af;border-bottom:2px solid #1e40af;padding-bottom:6px;margin-top:24px">
  📊 포트폴리오 기업 분기별 실적
  <small style="color:#6b7280;font-size:12px;font-weight:400">
    (최근 {len(next(iter(earn_results.values()))["quarters"])}개 분기 | 매출·영업이익·YoY 성장률)
  </small>
</h3>
{earn_sections}
<p style="font-size:11px;color:#9ca3af;margin-top:8px">
  ※ 지역별 매출(국내/해외)은 SEC EDGAR XBRL 공시 기업에 한해 표시 | yfinance quarterly_income_stmt 기준
</p>"""

    # ── 벨류체인 투자 현황 HTML ───────────────
    vc_html = ""
    if vc_results:
        vc_sections = ""
        from collections import defaultdict
        vc_by_parent = defaultdict(list)
        for r in vc_results:
            vc_by_parent[r["parent"]].append(r)

        for parent, rows in vc_by_parent.items():
            accent  = VALUE_CHAIN_DB[parent]["accent"]
            display = VALUE_CHAIN_DB[parent]["display"]
            vc_rows_html = ""
            for i, r in enumerate(rows):
                bg = "#f8faff" if i % 2 == 0 else "white"
                # 투자금액 배지
                inv_badge = (f'<span style="background:#fef3c7;color:#92400e;'
                             f'font-weight:700;padding:1px 6px;border-radius:8px;font-size:11px">'
                             f'{r["inv_fmt"]}</span>') if r["inv_usd"] else                             '<span style="color:#9ca3af;font-size:11px">—</span>'
                # 관계유형 배지
                rel_c = ("#16a34a" if "투자" in r["rel_type"] else
                         "#0891b2" if "공급" in r["rel_type"] or "파운드리" in r["rel_type"] else
                         "#7c3aed" if "파트너" in r["rel_type"] else
                         "#d97706")
                rel_badge = (f'<span style="background:{rel_c}20;color:{rel_c};'
                             f'font-weight:600;padding:1px 6px;border-radius:8px;font-size:11px">'
                             f'{r["rel_type"]}</span>')
                # 가격/수익률
                price_s = f'${r["price"]:.2f}' if r["price"] else '<span style="color:#9ca3af">비상장</span>'
                def _ret_cell(v):
                    if v is None: return '<span style="color:#9ca3af">N/A</span>'
                    c = "#16a34a" if v > 0 else "#dc2626"
                    return f'<span style="color:{c};font-weight:600">{"▲" if v>0 else "▼"}{abs(v):.1f}%</span>'
                rsi_s = (f'<span style="color:{"#dc2626" if r["rsi"]>70 else "#3b82f6" if r["rsi"]<30 else "#374151"}">'
                         f'{r["rsi"]:.1f}</span>') if r["rsi"] else '<span style="color:#9ca3af">—</span>'
                ticker_s = (f'<b style="color:#1e40af">{r["ticker"]}</b>'
                            if r["ticker"] != "비상장"
                            else '<span style="color:#9ca3af">비상장</span>')
                mc_s = r["market_cap"] or '<span style="color:#9ca3af">—</span>'

                vc_rows_html += f"""<tr style="background:{bg}">
  <td style="padding:11px 14px;font-size:12px;color:#374151">{r["area"]}</td>
  <td style="padding:11px 14px;font-weight:600">{r["company"]}</td>
  <td style="padding:11px 14px;text-align:center">{ticker_s}</td>
  <td style="padding:11px 14px;text-align:center">{inv_badge}</td>
  <td style="padding:11px 14px;text-align:center">{rel_badge}</td>
  <td style="padding:11px 14px;text-align:right;font-weight:600">{price_s}</td>
  <td style="padding:11px 14px;text-align:center">{rsi_s}</td>
  <td style="padding:11px 14px;text-align:center">{_ret_cell(r["ret_1m"])}</td>
  <td style="padding:11px 14px;text-align:center">{_ret_cell(r["ret_3m"])}</td>
  <td style="padding:11px 14px;text-align:center">{_ret_cell(r["ret_6m"])}</td>
  <td style="padding:11px 14px;text-align:center;font-size:11px">{mc_s}</td>
  <td style="padding:11px 14px;font-size:11px;color:#6b7280">{r["detail"]}</td>
  <td style="padding:11px 14px;text-align:center;font-size:11px;color:#374151;white-space:nowrap">{r.get("inv_date","—")}</td>
  <td style="padding:11px 14px;font-size:11px;color:#6b7280">{r.get("source","—")}</td>
</tr>"""

            vc_sections += f"""
<h4 style="color:{accent};margin:18px 0 6px;font-size:15px">
  {'🟢' if parent=='NVIDIA' else '🔵' if parent=='GOOGLE' else '🚀' if parent=='SPACEX' else '🔴'} {display}
</h4>
<table style="width:100%;border-collapse:collapse;font-size:13px;margin-bottom:12px">
<thead><tr style="background:{accent};color:white">
  <th style="padding:11px 14px;text-align:left">기술/분야</th>
  <th style="padding:11px 14px;text-align:left">투자·공급 기업</th>
  <th style="padding:11px 14px;text-align:center">티커</th>
  <th style="padding:11px 14px;text-align:center">투자금액</th>
  <th style="padding:11px 14px;text-align:center">관계유형</th>
  <th style="padding:11px 14px;text-align:right">현재가</th>
  <th style="padding:11px 14px;text-align:center">RSI</th>
  <th style="padding:11px 14px;text-align:center">1달</th>
  <th style="padding:11px 14px;text-align:center">분기</th>
  <th style="padding:11px 14px;text-align:center">반기</th>
  <th style="padding:11px 14px;text-align:center">시총</th>
  <th style="padding:11px 14px;text-align:left">설명</th>
  <th style="padding:11px 14px;text-align:center;white-space:nowrap">투자시점</th>
  <th style="padding:11px 14px;text-align:left">출처</th>
</tr></thead><tbody>{vc_rows_html}</tbody></table>"""

        vc_html = f"""
<h3 style="color:#1e293b;border-bottom:2px solid #1e293b;padding-bottom:6px;margin-top:24px">
  📡 벨류체인 투자 현황 — NVIDIA / Google / SpaceX / Tesla
  <small style="color:#6b7280;font-size:12px;font-weight:400">
    (전략투자·공급망·파트너십 | yfinance 실시간)
  </small>
</h3>
{vc_sections}
<p style="font-size:11px;color:#9ca3af;margin-top:8px;line-height:1.8">
  ※ 투자금액은 공개 보도 기준 (미공개 시 —) | 비상장 기업은 가격정보 없음<br>
  ※ 관계유형: 🟢 전략투자(지분취득) | 🔵 공급사(매출의존) | 🟣 전략파트너십
</p>"""

    html = f"""<html><body style="font-family:Arial,sans-serif;max-width:1500px;
margin:0 auto;padding:16px">
<h2 style="color:#1e40af;border-bottom:3px solid #1e40af;padding-bottom:8px">
  📈 주식 분석 리포트 — {today}
</h2>
{tw_banner}
{vix_banner}
{sector_html}
{asset_overview_html}
{industry_html}
{scan_section}
{val_section}
{fund_scan_section}
{fund_scan_footnote}
{port_section}
{fund_port_section}
{fund_port_footnote}
{earnings_html}
{vc_html}
{chart_section}
<p style="color:#9ca3af;font-size:12px;margin-top:24px;
  border-top:1px solid #e5e7eb;padding-top:10px">
  자동 생성 | {datetime.now().strftime("%Y-%m-%d %H:%M KST")} | 투자 판단은 본인 책임
</p></body></html>"""

    try:
            msg = MIMEMultipart("related")
            msg["Subject"] = (f"📈 주식 시그널 {today} "
                              f"— 스캔 {len(scan_results)}종목 / "
                              f"포트폴리오 {len(port_results)}종목")
            msg["From"] = EMAIL_FROM
            msg["To"]   = EMAIL_TO if isinstance(EMAIL_TO, str) \
                          else ", ".join(EMAIL_TO)

            import re
            from email.mime.image import MIMEImage

            cid_map = {}
            def replace_b64(match):
                b64_data = match.group(1)
                cid = f"chart_{len(cid_map)}"
                cid_map[cid] = b64_data
                return f'cid:{cid}'

            html_cid = re.sub(
                r'data:image/png;base64,([A-Za-z0-9+/=]+)',
                replace_b64, html)

            alt = MIMEMultipart("alternative")
            alt.attach(MIMEText(html_cid, "html"))
            msg.attach(alt)

            import base64 as b64lib
            for cid, b64_data in cid_map.items():
                img_data = b64lib.b64decode(b64_data)
                mime_img = MIMEImage(img_data, _subtype="png")
                mime_img.add_header("Content-ID", f"<{cid}>")
                mime_img.add_header("Content-Disposition", "inline",
                                    filename=f"{cid}.png")
                msg.attach(mime_img)

            with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
                server.login(EMAIL_FROM, EMAIL_PASSWORD)
                server.sendmail(EMAIL_FROM, EMAIL_TO, msg.as_string())
            print(f"\n  ✅ 이메일 발송 완료 → {EMAIL_TO}")
    except Exception as e:
        print(f"\n  ⚠ 이메일 발송 실패: {e}")

# ════════════════════════════════════════════
#  7. 출력 함수
# ════════════════════════════════════════════

def run_earnings_scan(port_results, n_quarters=11):
    """
    포트폴리오 종목별 분기 실적 수집 (yfinance quarterly_financials)
    반환: {ticker: {"company": str, "quarters": [...], "has_geo": bool}}
    quarters 항목: {"label": "24Q1", "rev": float|None, "dom": None,
                    "intl": None, "op_inc": float|None, "op_margin": float|None}
    단위: 백만달러(M)
    """
    if not port_results:
        return {}
    import yfinance as yf
    import pandas as pd

    tickers = [r["ticker"] for r in port_results if r.get("ticker")]
    print(f"\n  [분기 실적] {len(tickers)}개 종목 수집 중...")
    results = {}

    for ticker in tickers:
        try:
            t = yf.Ticker(ticker)
            info = t.info or {}
            company = info.get("shortName") or info.get("longName") or ticker

            fin = t.quarterly_financials   # index=항목명, columns=날짜(내림차순)
            if fin is None or fin.empty:
                results[ticker] = {"company": company, "quarters": [], "has_geo": False}
                continue

            cols = fin.columns.sort_values(ascending=False)[:n_quarters]

            def _find_row(keys):
                for k in keys:
                    for idx in fin.index:
                        if k.lower() in str(idx).lower():
                            return fin.loc[idx]
                return None

            rev_s   = _find_row(["Total Revenue", "Revenue"])
            op_s    = _find_row(["Operating Income", "EBIT"])

            def _val(series, col):
                if series is None:
                    return None
                try:
                    v = float(series[col])
                    if pd.isna(v):
                        return None
                    return round(v / 1e6, 1)
                except Exception:
                    return None

            quarters = []
            for col in cols:
                qnum  = (col.month - 1) // 3 + 1
                label = col.strftime("%y") + f"Q{qnum}"
                rev     = _val(rev_s, col)
                op_inc  = _val(op_s,  col)
                op_margin = None
                if rev and op_inc is not None and rev != 0:
                    op_margin = round(op_inc / rev * 100, 1)
                quarters.append({
                    "label":     label,
                    "rev":       rev,
                    "dom":       None,
                    "intl":      None,
                    "op_inc":    op_inc,
                    "op_margin": op_margin,
                })

            results[ticker] = {
                "company":  company,
                "quarters": quarters,
                "has_geo":  False,
            }
            print(f"    {ticker}: {len(quarters)}분기 수집 완료")
        except Exception as e:
            print(f"    {ticker}: 실적 수집 오류 — {e}")
            results[ticker] = {"company": ticker, "quarters": [], "has_geo": False}

    print(f"  [분기 실적] 완료")
    return results


def print_earnings_summary(earn_results):
    """분기 실적 텍스트 요약 출력"""
    if not earn_results:
        return
    print(f"\n{'='*60}")
    print(f"  분기별 실적 요약 ({len(earn_results)}개 종목)")
    print(f"{'='*60}")
    for ticker, d in earn_results.items():
        qs = d.get("quarters", [])
        if not qs:
            print(f"  {ticker}: 데이터 없음")
            continue
        latest = qs[0]
        rev_v = latest.get("rev")
        opm_v = latest.get("op_margin")
        rev_str = f"${rev_v:.0f}M"   if rev_v  is not None else "N/A"
        opm_str = f"{opm_v:.1f}%"    if opm_v  is not None else "N/A"
        print(f"  {ticker:6s}  최근분기 매출 {rev_str:>10s}  영업이익률 {opm_str}")


def display_earnings_table(earn_results):
    """Jupyter 환경에서 분기 실적 HTML 테이블 출력"""
    if not earn_results:
        return
    try:
        from IPython.display import display, HTML
        sections = ""
        for ticker, d in earn_results.items():
            qs = d.get("quarters", [])
            if not qs:
                continue
            ths = "".join(
                f'<th style="padding:6px 10px;background:#1e40af;color:white;'
                f'text-align:right;white-space:nowrap">{q["label"]}</th>'
                for q in qs
            )
            def _cell(v, pct=False):
                _na = '<td style="padding:6px 10px;color:#9ca3af;text-align:right">—</td>'
                if v is None:
                    return _na
                if pct:
                    color = "#16a34a" if v > 0 else "#dc2626" if v < 0 else "#6b7280"
                    return (f'<td style="padding:6px 10px;text-align:right;'
                            f'color:{color}">{v:+.1f}%</td>')
                return f'<td style="padding:6px 10px;text-align:right">${v:,.0f}M</td>'

            rev_cells = "".join(_cell(q["rev"])       for q in qs)
            op_cells  = "".join(_cell(q["op_inc"])    for q in qs)
            opm_cells = "".join(_cell(q["op_margin"], pct=True) for q in qs)

            sections += f"""
<h4 style="margin:20px 0 4px;color:#1e40af;font-size:15px">
  {ticker} — {d['company']}
</h4>
<table style="border-collapse:collapse;font-size:13px;width:100%;margin-bottom:8px">
<thead><tr>
  <th style="padding:6px 10px;background:#1e40af;color:white;text-align:left;min-width:120px">항목</th>
  {ths}
</tr></thead>
<tbody>
<tr style="background:#dbeafe">
  <td style="padding:6px 10px;font-weight:700">📊 매출 합계</td>{rev_cells}
</tr>
<tr style="background:#dcfce7">
  <td style="padding:6px 10px;font-weight:700">💰 영업이익</td>{op_cells}
</tr>
<tr style="background:#f9fafb">
  <td style="padding:6px 10px">영업이익률</td>{opm_cells}
</tr>
</tbody></table>"""

        display(HTML(
            '<div style="font-family:sans-serif;max-width:1400px">'
            '<h3 style="color:#1e293b;border-bottom:2px solid #1e293b;padding-bottom:6px">'
            '분기별 실적 추이</h3>'
            + sections + '</div>'
        ))
    except Exception as e:
        print(f"  [display_earnings_table] {e}")


def print_industry_summary(scan_results, port_results=None):
    """스캔·포트폴리오 결과를 산업별로 집계해 텍스트로 출력"""
    from collections import defaultdict
    ind_scan = defaultdict(list)
    for r in (scan_results or []):
        ind = r.get("industry", "") or "미분류"
        ind_scan[ind].append(r["ticker"])

    ind_port = defaultdict(list)
    for r in (port_results or []):
        ind = r.get("industry", "") or "미분류"
        ind_port[ind].append(r["ticker"])

    all_inds = sorted(set(list(ind_scan.keys()) + list(ind_port.keys())))
    if not all_inds:
        return

    print(f"\n{'='*60}")
    print(f"  📂 상세 산업별 현황")
    print(f"{'='*60}")
    print(f"  {'산업':35}  {'매수신호':>6}  {'보유중':>6}  종목")
    print("  " + "─"*90)
    for ind in all_inds:
        sc = ind_scan.get(ind, [])
        pc = ind_port.get(ind, [])
        all_t = sorted(set(sc + pc))
        scan_mark = f"{len(sc)}종목" if sc else "—"
        port_mark = f"{len(pc)}종목" if pc else "—"
        tickers_str = " ".join(all_t)[:40]
        print(f"  {ind[:35]:35}  {scan_mark:>6}  {port_mark:>6}  {tickers_str}")

def print_scan_results(scan_results):
    if not scan_results:
        print("  신호 발생 종목 없음")
        return
    print(f"\n  {'종목':>6}  {'전략':>4}  {'현재가':>8}  {'신호':>3}  "
          f"{'보정':>3}  {'주봉':>3}  {'섹터':>8}  {'추세':>6}  "
          f"{'손절가':>8}  {'목표가':>8}  R/R  {'세부산업':>28}  신호 상세")
    print("  " + "─"*108)

    # 출력 줄에 전략 추가
    for r in scan_results:
        sec   = r.get("sector","미분류")[:6]
        strat = r.get("strategy","C")
        print(f"  {r['ticker']:>6}  [{strat}]  ${r['price']:>7.2f}  "
              f"{r['signals']:>3}개  {r['adj_sig']:>3}개  {r['weekly']:>+3}  "
              f"{sec:>8}  {r.get('trend_type','미분류')[:4]:>6}  ${r['stop']:>7.2f}  ${r['target']:>7.2f}  "
              f"1:{r['rr']:.1f}  {r.get('industry','')[:28]:>28}  {r['details'][:25]}")

def print_portfolio_results(port_results):
    if not port_results:
        print("  포트폴리오 없음")
        return
    print(f"\n  {'종목':>6}  {'매수가':>7}  {'현재가':>7}  {'수익률':>7}  "
          f"{'손절가':>7}  {'목표가':>7}  {'신호':>3}  상태")
    print("  " + "─"*75)
    for r in port_results:
        print(f"  {r['ticker']:>6}  ${r['buy_price']:>6.2f}  "
              f"${r['cur_price']:>6.2f}  {r['pnl_pct']:>+6.2f}%  "
              f"${r['stop']:>6.2f}  ${r['target']:>6.2f}  "
              f"{r['signals']:>3}개  {r['status']}")

# ════════════════════════════════════════════
#  8. 통합 실행
# ════════════════════════════════════════════
def daily_job():
    import json, os
    print("  ✅ PATCH v3 로드 확인 — 산업별 현황 + 404억제 활성")

    # 섹터 분석
    sector_data = analyze_sectors()
    print_sector_summary(sector_data)

    # 섹터 맵 캐시 로드
    # 신포맷(industry 포함) 우선, 없으면 구포맷 폴백
    if os.path.exists("sector_map_industry.json"):
        with open("sector_map_industry.json", "r", encoding="utf-8") as f:
            sector_map = json.load(f)
        print(f"  섹터 맵 로드 (산업 세분화): {len(sector_map)}종목")
    elif os.path.exists("sector_map_cache.json"):
        with open("sector_map_cache.json", "r") as f:
            sector_map = json.load(f)
        print(f"  섹터 맵 로드 (기본): {len(sector_map)}종목")
    else:
        sector_map = {}
        print("  ⚠ sector_map 없음 — 섹터 가중치 미적용")

    scan_res = run_market_scan(sector_data, sector_map)
    print_scan_results(scan_res)

    # ── 통합 자산 로드 (현금/예적금/원자재/주식)
    unified_portfolio = load_unified_portfolio()

    port_res = run_portfolio_update(sector_map, sector_data)
    print_portfolio_results(port_res)
    print_industry_summary(scan_res, port_res)

    vc_res = run_value_chain_scan()
    print_value_chain_results(vc_res)

    # ── 분기별 실적 분석 ──
    earn_res = run_earnings_scan(port_res)
    print_earnings_summary(earn_res)
    display_earnings_table(earn_res)

    today = datetime.now().strftime("%Y%m%d")
    if scan_res:
        pd.DataFrame(scan_res).to_csv(
            f"scan_result_{today}.csv", index=False, encoding="utf-8-sig")
        print(f"\n  스캔 결과 저장: scan_result_{today}.csv")
    if port_res:
        pd.DataFrame(port_res).to_csv(
            f"portfolio_update_{today}.csv", index=False, encoding="utf-8-sig")
        print(f"  포트폴리오 저장: portfolio_update_{today}.csv")

    if EMAIL_FROM != "your_email@gmail.com":
        send_email(scan_res, port_res, sector_data, vc_res, earn_res, sector_map=sector_map, unified_portfolio=unified_portfolio)
    else:
        print("\n  ⚠ 이메일 미설정")

# ════════════════════════════════════════════
#  9. 즉시 실행 (1회)
# ════════════════════════════════════════════
daily_job()


  ✅ PATCH v3 로드 확인 — 산업별 현황 + 404억제 활성

  [섹터 분석] SPDR ETF 11개 로딩 중...


  섹터 분석 완료 (11개)

    ETF  섹터                              주봉    RSI      1주      1달      분기      반기      추세유형  상태
  ───────────────────────────────────────────────────────────────────────────────────────────────
    XLK  기술 (Technology)                 +3   80.4   ▲6.7%  ▲21.4%  ▲40.9%  ▲37.8%      완전상승  🟡 중립
    XLI  산업재 (Industrials)               +3   49.3   ▼0.1%   ▲1.8%   ▼0.8%  ▲15.7%        혼조  🟢 강세
    XLB  소재 (Materials)                  +3   45.3   ▲0.7%   ▲1.8%   ▼0.3%  ▲16.7%      상승추세  🟢 강세
    XLY  임의소비재 (Consumer Discretionary)    +3   49.2   ▼2.9%   ▲0.3%   ▲1.6%   ▲0.2%      상승추세  🟢 강세
   XLRE  부동산 (Real Estate)               +3   37.0   ▼2.7%   ▼1.5%   ▼0.1%   ▲7.6%      약세전환  🟡 중립
    XLE  에너지 (Energy)                    +3   50.0   ▲1.0%   ▼3.0%   ▲3.1%  ▲28.0%      상승추세  ⚪ 관망
    XLV  헬스케어 (Health Care)              +1   52.2   ▼1.5%   ▲1.2%   ▼6.3%   ▼4.8%      약세전환  🟡 중립
    XLC  커뮤니케이션 (Communication)          +1   41.4   ▼1.6%   ▼1.5%   ▼3.4%   ▲0.2%      약세전환

  ⚠ S&P500 Wikipedia 파싱 실패: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.
  ⚠ S&P500 자동 로드 전체 실패 — 하드코딩 사용 (130종목)
  ⚠ 나스닥100 Wikipedia 파싱 실패: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.
  ⚠ 나스닥100 자동 로드 전체 실패 — 하드코딩 사용 (50종목)
  스캔 대상: 145종목



  스캔 완료 → 신호 발생: 2종목 (오류: 0)

      종목    전략       현재가   신호   보정   주봉        섹터      추세       손절가       목표가  R/R                          세부산업  신호 상세
  ────────────────────────────────────────────────────────────────────────────────────────────────────────────
    GOOG  [C]  $ 369.24    5개    8개   +3      🟡 중립    약세전환  $ 353.22  $ 401.27  1:2.0  Internet Content & Informati  구름대 위 / Stoch GC(K=32) / 
      ON  [C]  $ 129.80    5개    8개   +3      🟡 중립    완전상승  $ 115.61  $ 158.18  1:2.0                Semiconductors  구름대 위 / MA 정배열 / MACD 양전환


  [통합자산] 주식 22종목 / 원자재 1종목 로드


...

    COHR: 6분기 수집 완료


    NVDA: 5분기 수집 완료


    RKLB: 5분기 수집 완료
  [분기 실적] 완료

  분기별 실적 요약 (9개 종목)
  RKLX: 데이터 없음
  GOOGL   최근분기 매출   $109896M  영업이익률 36.1%
  TSLL: 데이터 없음
  CDNS    최근분기 매출     $1474M  영업이익률 29.3%
  MPWR    최근분기 매출      $804M  영업이익률 30.0%
  TSLA    최근분기 매출    $22387M  영업이익률 4.2%
  COHR    최근분기 매출     $1806M  영업이익률 1.6%
  NVDA    최근분기 매출    $81615M  영업이익률 65.6%
  RKLB    최근분기 매출      $200M  영업이익률 -28.0%



  스캔 결과 저장: scan_result_20260602.csv
  포트폴리오 저장: portfolio_update_20260602.csv



  ✅ 이메일 발송 완료 → dltkdals15319@gmail.com
